# Comparativo CORREGIDO sobre **UCSD/McAuley** (protocolo paper 80/10/10, 1000 ep + `user_LLM` entrenable) — base · +PER · +PER+PRG

2º dataset (recupera el descuento single-dataset de H2 y habilita la **tabla cross-dataset**). Una sola corrida: entrena **CPGRec base**, **+PER** y **+PER+PRG** (= CPGRec+ completo), entrena **ALS** (techo colaborativo, estilo H2), y produce 4 análisis: **§A accuracy**, **§B diversidad**, **§C long-tail por actividad**, **§D ejemplos para el póster**. Las 3 variantes GNN se entrenan en la **misma sesión** → deltas **pareados** válidos. Guarda solo resultados chicos (CSV/JSON/md).

Espeja **byte-idéntico** el notebook canónico de SCGRec (`CPGRec_comparativo_multiseed_T2_corregidoejecutado.ipynb`); solo cambian la capa de datos (UCSD `ucsd_ready/`), el rating de PER (positive_ratio de reviews), el texto SBERT (+tags), el filtro (5-core, sin subsample) y `REPORTED={}` (UCSD no es dataset de los papers CPGRec/CPGRec+).

**Setup DGL (al inicio):** *Ejecutar todo* → la 1ª celda de código (setup) reinicia el runtime **al instante** → *Ejecutar todo* de nuevo y corre de corrido. Siempre *Ejecutar todo desde arriba* (no "Ejecutar después").

## 0. Setup del entorno GPU (DGL + torch 2.4) — CORRER ESTA CELDA PRIMERO

(portado VERBATIM de H2; reubicado al inicio para que el reinicio del runtime ocurra ANTES de cargar datos. Flujo: "Ejecutar todo" -> reinicia al instante -> "Ejecutar todo" de nuevo.)

In [ ]:
# transformers/sentence-transformers PINEADOS a versiones compatibles con torch 2.4
# (DGL exige torch 2.4; el transformers reciente importa torch.distributed.tensor.device_mesh,
#  que NO existe en torch 2.4 -> rompe el import de SBERT). NO subir torch (rompe DGL).
!pip install -q kagglehub "transformers==4.44.2" "sentence-transformers==3.0.1" implicit

import importlib, importlib.util, torch

def _dgl_ok():
    importlib.invalidate_caches()
    return importlib.util.find_spec("dgl") is not None

torch_ok = torch.__version__.startswith("2.4")

if not torch_ok:
    print("Fijando torch 2.4 (compatible con el graphbolt de DGL)...")
    !pip install -q torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu118

if not _dgl_ok():
    !pip uninstall -y dgl 2>/dev/null
    !pip install -q dgl -f https://data.dgl.ai/wheels/torch-2.4/cu118/repo.html
    !pip install -q torchdata==0.7.1 --no-deps

assert _dgl_ok(), "DGL no se instaló: revisá la salida de pip de arriba."

# transformers CARGADO en memoria debe ser 4.44 (si Colab pre-importo uno nuevo, pip lo bajo en
# disco pero el modulo viejo sigue en memoria -> hay que REINICIAR para que SBERT no rompa con
# torch 2.4: el transformers reciente importa torch.distributed.tensor.device_mesh, inexistente en 2.4).
import transformers
tf_ok = transformers.__version__.startswith("4.44")

if not torch_ok or not tf_ok:
    print(f"\n*** Reiniciando runtime para aplicar torch 2.4 / transformers 4.44 "
          f"(torch={torch.__version__}, transformers={transformers.__version__}). "
          f"Al reiniciar: Ejecutar todo de nuevo. ***")
    import IPython; IPython.Application.instance().kernel.do_shutdown(True)
else:
    import dgl; print("dgl", dgl.__version__, "| torch", torch.__version__,
                      "| transformers", transformers.__version__, "OK")

DGL backend not selected or invalid.  Assuming PyTorch for now.


Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)
dgl 2.4.0+cu118 | torch 2.4.0+cu118 | transformers 4.44.2 OK


> **Nota:** la celda de arriba **reinicia el runtime** (para fijar torch 2.4 + DGL +
> transformers 4.44; un transformers mas nuevo rompe SBERT con torch 2.4). Es esperado: cuando reinicie, simplemente vuelve a *Ejecutar todo*. La segunda vez la
> celda ve que torch ya es 2.4 e importa DGL sin reinstalar nada. Si tras el reinicio el
> `import dgl` aún fallara, probar `cu118` en lugar de `cu121` en las dos líneas de install.

In [ ]:
!pip uninstall -y torchcodec

In [ ]:
# ===================== CPGRec corregido sobre UCSD/McAuley (protocolo paper 80/10/10) =====================
# Espeja el comparativo corregido de SCGRec, pero sobre UCSD/McAuley (2o dataset):
#  - protocolo del paper: re-split RANDOM 80/10/10, full-ranking, paper_metrics multi-rel @{5,10}
#  - config CORREGIDA: 1000 epocas + user_embedding_LLM ENTRENABLE (--freeze_user_llm 0) + 3 seeds
#  - PER alimentado con positive_ratio de reviews (fraccion recommend=True por juego; rating poblado -> senal real)
#  - PRG-SBERT: texto = nombre + generos + dev + pub + TAGS (UCSD trae tags)
#  - filtro 5-core (paralelo al filtro de CPGRec sobre SCGRec); UCSD es chico -> SIN subsample de usuarios
import os, sys, glob, subprocess, json, math, time
import numpy as np
import pandas as pd

REPO_URL = 'https://github.com/Benjaa7/Proyecto-RecSys.git'   # <-- ajusta si tu repo cambia
REPO_DIR = '/content/Proyecto-RecSys'
def _locate_protocol():
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        subprocess.run(['git', '-C', REPO_DIR, 'fetch', '-q', '--depth', '1', 'origin'], check=False)
        subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', '-q', 'FETCH_HEAD'], check=False)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', '-q', REPO_URL, REPO_DIR], check=False)
    h3 = os.path.join(REPO_DIR, 'H3')
    if os.path.exists(os.path.join(h3, 'recsys_protocol.py')):
        return h3
    for c in ['/content', '.', '..'] + sorted(glob.glob('/content/*')):
        if os.path.exists(os.path.join(c, 'recsys_protocol.py')):
            return c
    return None
_root = _locate_protocol()
if _root:
    sys.path.insert(0, _root)
    for _m in [m for m in list(sys.modules) if m == 'recsys_protocol' or m.startswith('recsys_protocol.')]:
        del sys.modules[_m]
    from recsys_protocol import set_global_seed, SEED
    set_global_seed()
else:
    SEED = 42
    def set_global_seed(seed=42, **k):
        import random; random.seed(seed); np.random.seed(seed)
    set_global_seed()
    print('[warn] recsys_protocol no encontrado; seed local 42')

# -- Hiperparametros del PAPER (CPGRec, Apendice A) --
EMBED_SIZE, LR, BATCH, M_NSR, PARAM_DECAY = 32, 0.03, 1024, 6.5, 0.1
KS = (5, 10)
EMB_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'   # SBERT 384-dim (rama +PRG-SBERT)
PRG_EMB_DIM = 64   # PCA del SBERT (evita OOM del user_embedding_LLM; bajo riesgo en UCSD por ser chico)
USE_LLM = 1        # el comparativo siempre genera el SBERT; cada corrida fija --use_llm 0/1 aparte

# -- ALS (techo colaborativo, estilo H2: confianza = 1 + alpha*playtime) --
ALS_FACTORS, ALS_ITERS, ALS_REG, ALS_ALPHA = 64, 15, 0.1, 40.0

# -- Analisis comparativo --
TOPN            = 10
N_EVAL_ANALYSIS = 200_000     # submuestra de eval (None=todos); UCSD es chico -> normalmente evalua a TODOS
N_EXAMPLES      = 3
SEEDS           = [42, 1, 2]   # 3 seeds en UNA corrida (split/eval fijos con SEED) -> media+-std
FREEZE_USER_LLM = 0            # CORREGIDO: 0 = user_embedding_LLM entrenable (como el repo)

# -- UCSD/McAuley (dataset + filtrado) --
UCSD_DIR  = 'ucsd_ready'
MIN_USER, MIN_ITEM = 5, 5      # 5-core: >=5 interac/usuario y >=5 interac/juego (preserva long-tail §C)
OR_CAP    = 50                 # tope grado G^Co. or_cap=0 (sin tope) = producto cuadratico exacto -> OOM en UCSD denso (9.198 juegos, grupos 'Indie' enormes). 50 = igual que Kozyriev.
OR_WEIGHT = 'size'

TIER = 'T2'
# UCSD es el mas liviano de los 3 datasets (~60k usuarios 5-core) -> 9 entrenamientos x 1000 ep
# entran en UNA sesion SIN subsample. T1 = sanity (50 ep). SUBSAMPLE_USERS=None en ambos tiers.
SUBSAMPLE_USERS, EPOCHS = (None, 1000) if TIER == 'T2' else (None, 50)
print(f'UCSD corregido | TIER={TIER} | emb={EMBED_SIZE} epochs={EPOCHS} subsample={SUBSAMPLE_USERS} | '
      f'5-core=({MIN_USER},{MIN_ITEM}) or_cap={OR_CAP} | freeze_user_llm={FREEZE_USER_LLM} | SEEDS={SEEDS}')

[protocol] v2026-06-23b (defaults: eval sin tope max_eval_users=None + frac_train=0.30)
[protocol] seed global = 42 | numpy/random/torch (cuda=True, determinista=True)
UCSD corregido | TIER=T2 | emb=32 epochs=1000 subsample=None | 5-core=(5,5) or_cap=50 | freeze_user_llm=0 | SEEDS=[42, 1, 2]


## 0.5 Bootstrap de datos (autocontenido)

Si faltan `ucsd_ready/*.parquet`, esta celda **descarga y parsea UCSD/McAuley sola** (espeja
`load_ucsd.ipynb`): los `.json.gz` de McAuley son dicts de Python por línea (`ast.literal_eval`),
leídos en streaming sobre el gzip remoto. Escribe `inter`/`game_categories`/`reviews` con el
**mismo esquema-lista que `scgrec_ready/`** → reusa el pipeline. Así el notebook corre con
*Ejecutar todo* desde cero. Si ya subiste/corriste los parquets, se salta.

In [ ]:
# ====== Bootstrap: descarga + parsea UCSD/McAuley si faltan los parquets (autocontenido) ======
_need_parq = [f'{UCSD_DIR}/inter.parquet', f'{UCSD_DIR}/game_categories.parquet', f'{UCSD_DIR}/reviews.parquet']
if all(os.path.exists(p) for p in _need_parq):
    print('parquets ya presentes:', _need_parq)
else:
    import ast, gzip, urllib.request
    os.makedirs(UCSD_DIR, exist_ok=True)
    _H = {'User-Agent': 'Mozilla/5.0'}
    _URLS = {
        'steam_games':  'https://cseweb.ucsd.edu/~wckang/steam_games.json.gz',
        'users_items':  'https://mcauleylab.ucsd.edu/public_datasets/data/steam/australian_users_items.json.gz',
        'user_reviews': 'https://mcauleylab.ucsd.edu/public_datasets/data/steam/australian_user_reviews.json.gz',
    }
    def _stream(url):
        print('descargando', url)
        req = urllib.request.Request(url, headers=_H); rows = []
        with urllib.request.urlopen(req) as resp, gzip.GzipFile(fileobj=resp) as gz:
            for line in gz:
                rows.append(ast.literal_eval(line.decode('utf-8')))
        return rows
    def _as_list(x):
        if isinstance(x, (list, tuple)): return [str(v).strip() for v in x if v not in (None, '')]
        if x is None: return []
        if isinstance(x, float) and pd.isna(x): return []
        s = str(x).strip(); return [s] if s else []
    def _to_int(x):
        try: return int(x)
        except (TypeError, ValueError): return None
    def _games_to_cat(games):
        rec = []
        for g in games:
            aid = _to_int(g.get('id'))
            if aid is None: continue
            rec.append({'app_id': aid, 'genres': _as_list(g.get('genres')),
                        'developers': _as_list(g.get('developer')), 'publishers': _as_list(g.get('publisher')),
                        'name': g.get('app_name') or g.get('title') or 'null', 'price': g.get('price'),
                        'release_date': g.get('release_date'), 'sentiment': g.get('sentiment'),
                        'metascore': g.get('metascore'), 'tags': _as_list(g.get('tags'))})
        out = pd.DataFrame(rec).drop_duplicates('app_id').reset_index(drop=True)
        # columnas de tipos mixtos -> str para que pyarrow no falle al escribir parquet
        for c in ['price', 'sentiment', 'metascore', 'release_date']:
            out[c] = out[c].astype(str)
        return out
    def _users_to_inter(users):
        u, a, p = [], [], []
        for usr in users:
            uid = usr.get('user_id')
            for it in (usr.get('items') or []):
                aid = _to_int(it.get('item_id'))
                if uid is None or aid is None: continue
                pt = it.get('playtime_forever')
                u.append(str(uid)); a.append(aid); p.append(float(pt) if pt is not None else 0.0)
        return pd.DataFrame({'user_id': u, 'app_id': a, 'playtime': p})
    def _reviews_to_df(users):
        rec = []
        for usr in users:
            uid = usr.get('user_id')
            for r in (usr.get('reviews') or []):
                aid = _to_int(r.get('item_id'))
                if uid is None or aid is None: continue
                rc = r.get('recommend')
                rec.append({'user_id': str(uid), 'app_id': aid,
                            'recommend': (bool(rc) if rc is not None else None)})
        return pd.DataFrame(rec)
    if not os.path.exists(_need_parq[0]):
        _users_to_inter(_stream(_URLS['users_items'])).to_parquet(_need_parq[0], index=False)
    if not os.path.exists(_need_parq[1]):
        _games_to_cat(_stream(_URLS['steam_games'])).to_parquet(_need_parq[1], index=False)
    if not os.path.exists(_need_parq[2]):
        _reviews_to_df(_stream(_URLS['user_reviews'])).to_parquet(_need_parq[2], index=False)
    print('parquets listos:', _need_parq)

parquets ya presentes: ['ucsd_ready/inter.parquet', 'ucsd_ready/game_categories.parquet', 'ucsd_ready/reviews.parquet']


## 1. Datos: 5-core + factorize user_id + re-split RANDOM 80/10/10

UCSD comparte el esquema-lista de `scgrec_ready/`, con dos diferencias que se manejan acá: (a) el
`user_id` es **string** (vanity names) → se **factoriza** a entero contiguo (el escritor `steam_data`
y ALS asumen id entero); (b) el rating de **PER** se alimenta con el **positive_ratio de reviews**
(fracción `recommend=True` por juego, puesto en `cat['metascore']`). Filtro **5-core**; UCSD es chico
→ **sin subsample**. Split **random 80/10/10** (no LOO temporal: `australian_users_items` no trae fecha
por interacción → split "a la manera de los autores", caveat de protocolo del paper).

In [ ]:
# ====== Carga UCSD (parquets) + 5-core + factorize user_id + positive_ratio + re-split 80/10/10 ======
def _need(p):
    if not os.path.exists(p):
        raise FileNotFoundError(f'No encuentro {p}. Corre la celda de bootstrap o load_ucsd.ipynb.')
    return p
inter   = pd.read_parquet(_need(f'{UCSD_DIR}/inter.parquet'))
cat_raw = pd.read_parquet(_need(f'{UCSD_DIR}/game_categories.parquet'))
reviews = pd.read_parquet(_need(f'{UCSD_DIR}/reviews.parquet'))
inter['app_id']  = pd.to_numeric(inter['app_id'], errors='coerce')
inter = inter.dropna(subset=['app_id']); inter['app_id'] = inter['app_id'].astype('int64')
inter['user_id'] = inter['user_id'].astype(str)
print(f'crudo: {inter["user_id"].nunique():,} usuarios c/interac · {inter["app_id"].nunique():,} juegos · {len(inter):,} interac')

# 5-core iterativo: >=MIN_USER interac/usuario y >=MIN_ITEM interac/juego
def iterative_k_core(df, min_user, min_item):
    it = 0
    while True:
        n0 = len(df)
        uc = df['user_id'].value_counts(); ic = df['app_id'].value_counts()
        df = df[df['user_id'].isin(uc.index[uc >= min_user]) & df['app_id'].isin(ic.index[ic >= min_item])]
        it += 1; print(f'  k-core iter {it}: {n0:,} -> {len(df):,}')
        if len(df) == n0 or len(df) == 0: return df
inter = iterative_k_core(inter, MIN_USER, MIN_ITEM).reset_index(drop=True)
print(f'5-core ({MIN_USER},{MIN_ITEM}): {inter["user_id"].nunique():,} usuarios · '
      f'{inter["app_id"].nunique():,} juegos · {len(inter):,} interac')

# subsample de usuarios (UCSD: None = todos)
if SUBSAMPLE_USERS is not None:
    _rng = np.random.default_rng(SEED)
    _u = np.sort(inter['user_id'].unique())
    keep = set(_rng.choice(_u, size=min(SUBSAMPLE_USERS, len(_u)), replace=False).tolist())
    inter = inter[inter['user_id'].isin(keep)].copy()

# factorize user_id (str -> int contiguo); guarda reverse-map para los ejemplos §D
_ucodes, _uuniques = pd.factorize(inter['user_id'], sort=True)
inter['user_id'] = _ucodes.astype('int64')
user_code2orig = {i: str(u) for i, u in enumerate(_uuniques)}   # int -> vanity id original

# re-split RANDOM 80/10/10 por interaccion (protocolo del paper)
_rng = np.random.default_rng(SEED)
n = len(inter); perm = _rng.permutation(n)
inter = inter.iloc[perm].reset_index(drop=True)
n_tr, n_va = int(0.8 * n), int(0.1 * n)
split = np.empty(n, dtype='int8'); split[:n_tr] = 0; split[n_tr:n_tr + n_va] = 1; split[n_tr + n_va:] = 2
inter['split'] = split
train, test = inter[inter['split'] == 0], inter[inter['split'] == 2]

# CATALOG = items con interaccion (5-core)
CATALOG = sorted(int(a) for a in inter['app_id'].unique()); catalog_set = set(CATALOG); n_catalog = len(CATALOG)

# PER rating: positive_ratio = fraccion recommend=True por juego (x100). Continuo y poblado -> PER con senal.
reviews['app_id'] = pd.to_numeric(reviews['app_id'], errors='coerce')
_rv = reviews.dropna(subset=['app_id']).copy(); _rv['app_id'] = _rv['app_id'].astype('int64')
_rv['recommend'] = _rv['recommend'].fillna(False).astype(bool)
_pr = (_rv.groupby('app_id')['recommend'].mean() * 100.0)
_cov = sum(1 for a in CATALOG if a in _pr.index) / max(1, n_catalog)
_pr_med = float(_pr.reindex(CATALOG).median()) if len(_pr) else 50.0
if not (_pr_med == _pr_med): _pr_med = 50.0   # NaN guard si ningun item del catalogo tiene review
print(f'positive_ratio (PER rating): {len(_pr):,} juegos con >=1 review | '
      f'cobertura del catalogo={_cov*100:.1f}% | mediana={_pr_med:.1f} (faltantes->mediana)')

# cat alineado a CATALOG, con fallbacks; metascore = positive_ratio
def _aslist(x):
    if isinstance(x, np.ndarray): return [v for v in x.tolist() if v is not None]
    if isinstance(x, list):       return [v for v in x if v is not None]
    return [] if x is None else [x]
_cx = cat_raw.drop_duplicates('app_id').set_index('app_id')
def _cv(aid, col, default=None):
    try:
        v = _cx.at[aid, col]
        if isinstance(v, (list, np.ndarray)): return v
        return v if (v is not None and not (isinstance(v, float) and pd.isna(v))) else default
    except Exception:
        return default
rows = []
for a in CATALOG:
    pr = _pr.get(a, np.nan)
    nm = _cv(a, 'name')
    rows.append({'app_id': a,
                 'name': (str(nm) if isinstance(nm, str) and nm.strip() else f'app_{a}'),
                 'price': pd.to_numeric(_cv(a, 'price'), errors='coerce'),
                 'release_date': _cv(a, 'release_date'),
                 'metascore': (float(pr) if pd.notna(pr) else _pr_med),   # PER rating = positive_ratio
                 'genres': _aslist(_cv(a, 'genres', [])),
                 'developers': _aslist(_cv(a, 'developers', [])),
                 'publishers': _aslist(_cv(a, 'publishers', [])),
                 'tags': _aslist(_cv(a, 'tags', []))})
cat = pd.DataFrame(rows)
_medp = pd.to_numeric(cat['price'], errors='coerce').median()
cat['price'] = pd.to_numeric(cat['price'], errors='coerce').fillna(_medp if pd.notna(_medp) else 0.0)

genre_map = {int(a): list(g) for a, g in zip(cat['app_id'], cat['genres'])}
dev_map   = {int(a): list(g) for a, g in zip(cat['app_id'], cat['developers'])}
pub_map   = {int(a): list(g) for a, g in zip(cat['app_id'], cat['publishers'])}
total_map = {a: [('g', x) for x in genre_map.get(a, [])] + [('d', x) for x in dev_map.get(a, [])]
                + [('p', x) for x in pub_map.get(a, [])] for a in CATALOG}
CAT_MAPS = {'gene': genre_map, 'dev': dev_map, 'pub': pub_map, 'total': total_map}

train_items_per_user = train.groupby('user_id')['app_id'].apply(set).to_dict()
test_items_per_user  = {int(u): set(int(x) for x in g) for u, g in test.groupby('user_id')['app_id']}
eval_users = [u for u in test_items_per_user if u in train_items_per_user]
_ntags = int(cat['tags'].map(lambda t: len(_aslist(t)) > 0).sum())
print(f'cat={len(cat):,} juegos | con genero={sum(1 for a in CATALOG if genre_map.get(a)):,} | con tags={_ntags:,}')
print(f'train={len(train):,} test={len(test):,} | usuarios train={train["user_id"].nunique():,} | '
      f'catalogo={n_catalog:,} | eval usuarios={len(eval_users):,} | '
      f'positivos/usuario (medio)={np.mean([len(test_items_per_user[u]) for u in eval_users]):.2f}')

crudo: 70,912 usuarios c/interac · 10,978 juegos · 5,153,209 interac
  k-core iter 1: 5,153,209 -> 5,132,610
  k-core iter 2: 5,132,610 -> 5,132,610
5-core (5,5): 62,944 usuarios · 9,198 juegos · 5,132,610 interac
positive_ratio (PER rating): 3,682 juegos con >=1 review | cobertura del catalogo=35.6% | mediana=100.0 (faltantes->mediana)
cat=9,198 juegos | con genero=7,449 | con tags=7,779
train=4,106,088 test=513,261 | usuarios train=62,944 | catalogo=9,198 | eval usuarios=57,390 | positivos/usuario (medio)=8.93


## Nombres de juegos (para los ejemplos)

In [ ]:
# Nombres de juegos (para los ejemplos del póster): app_id -> name desde game_categories.
_cx = cat.set_index('app_id')
def _gname(a):
    if a in _cx.index:
        v = _cx.loc[a].get('name')
        if isinstance(v, str) and v.strip(): return v
    return f'app_{a}'
name_map = {a: _gname(a) for a in CATALOG}
print('name_map listo:', len(name_map), 'juegos. Ej:', list(name_map.items())[:3])


name_map listo: 9198 juegos. Ej: [(10, 'Counter-Strike'), (20, 'Team Fortress Classic'), (30, 'Day of Defeat')]


## 2. Métricas del protocolo del paper

Accuracy multi-relevante (Recall/NDCG/Hit/Precision @{5,10}) — varios positivos por usuario.
**Coverage/Entropy a nivel CATEGORÍA** (§5.1.2): género/dev/pub (+ total); Coverage = nº medio
de categorías distintas en top-K (conteo); Entropy = entropía de Shannon media de la distribución.

In [ ]:
# Métricas del protocolo de reproducción (accuracy multi-relevante + diversidad por categoría).
def _dcg(hits):
    return sum((1.0 / math.log2(i + 2)) for i, h in enumerate(hits) if h)

def _cat_cov_ent(recs, cat_map, k):
    covs, ents = [], []
    for rec in recs.values():
        cnt = {}
        for it in rec[:k]:
            for c in cat_map.get(it, ()):
                cnt[c] = cnt.get(c, 0) + 1
        if not cnt:
            covs.append(0); ents.append(0.0); continue
        covs.append(len(cnt)); tot = sum(cnt.values())
        ents.append(-sum((v / tot) * math.log2(v / tot) for v in cnt.values()))
    return (float(np.mean(covs)) if covs else 0.0, float(np.mean(ents)) if ents else 0.0)

def paper_metrics(recs, test_items, ks=KS, cat_maps=None):
    acc = {f'{m}@{k}': [] for k in ks for m in ('Recall', 'NDCG', 'Hit', 'Precision')}
    n = 0
    for u, rec in recs.items():
        rel = test_items.get(u)
        if not rel:
            continue
        n += 1
        for k in ks:
            hits = [(1 if it in rel else 0) for it in rec[:k]]
            nhit = sum(hits)
            acc[f'Recall@{k}'].append(nhit / len(rel))
            acc[f'Precision@{k}'].append(nhit / k)
            acc[f'Hit@{k}'].append(1.0 if nhit > 0 else 0.0)
            idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(rel), k)))
            acc[f'NDCG@{k}'].append(_dcg(hits) / idcg if idcg > 0 else 0.0)
    out = {key: (float(np.mean(v)) if v else 0.0) for key, v in acc.items()}
    if cat_maps:
        for k in ks:
            for name, cmap in cat_maps.items():
                cov, ent = _cat_cov_ent(recs, cmap, k)
                out[f'Cov_{name}@{k}'] = cov; out[f'Ent_{name}@{k}'] = ent
    out['n_users'] = n
    return out

## 3. Baseline Most Popular (sanity del arnés + piso)

In [ ]:
pop = train.groupby('app_id').size().to_dict()
popular_list = [it for it, _ in sorted(pop.items(), key=lambda kv: (-kv[1], kv[0])) if it in catalog_set]

TOPN = max(KS)
recs_mp = {}
for u in eval_users:
    seen = train_items_per_user.get(u, set())
    recs_mp[u] = [i for i in popular_list if i not in seen][:TOPN]

m_mp = paper_metrics(recs_mp, test_items_per_user, cat_maps=CAT_MAPS)
print('Most Popular (split 80/10/10 UCSD):')
for k in KS:
    print(f'  @{k}: Recall={m_mp[f"Recall@{k}"]:.4f} NDCG={m_mp[f"NDCG@{k}"]:.4f} '
          f'Hit={m_mp[f"Hit@{k}"]:.4f} Prec={m_mp[f"Precision@{k}"]:.4f} '
          f'Cov(total)={m_mp[f"Cov_total@{k}"]:.2f} Ent(gene)={m_mp[f"Ent_gene@{k}"]:.3f}')

Most Popular (split 80/10/10 UCSD):
  @5: Recall=0.1170 NDCG=0.1652 Hit=0.4635 Prec=0.1244 Cov(total)=11.19 Ent(gene)=2.189
  @10: Recall=0.1599 NDCG=0.1633 Hit=0.5717 Prec=0.0900 Cov(total)=17.50 Ent(gene)=2.448


## Sin baseline de paper (UCSD no es dataset de CPGRec/CPGRec+)

In [ ]:
# UCSD/McAuley NO es dataset de los papers CPGRec/CPGRec+ (Cheuque'19 sobre UCSD usa ALS/FM/DeepFM,
# NO CPGRec; CPGRec/CPGRec+ corrieron sobre SCGRec + "Steam2") -> sin numeros REPORTADOS de referencia.
# REPORTED vacio: la tabla §A no agrega filas de paper (igual que en Kozyriev).
REPORTED = {}
ACC = [f'{x}@{k}' for k in KS for x in ('Recall','NDCG','Hit','Precision')]
def _row(m): return {c: m.get(c) for c in ACC}
print('Sin baseline de paper para UCSD; ACC =', ACC)

Sin baseline de paper para UCSD; ACC = ['Recall@5', 'NDCG@5', 'Hit@5', 'Precision@5', 'Recall@10', 'NDCG@10', 'Hit@10', 'Precision@10']


## 5. Entrenamiento (CPGRec base + +PRG-SBERT + ALS) — GPU

In [ ]:
import os, shutil
# Un clon es VÁLIDO solo si tiene utils/ (un CPGRec_plus/ vacío puede quedar de un intento
# fallido o de os.makedirs); en ese caso lo borramos y reclonamos. Así %%writefile (que NO
# crea directorios) siempre encuentra CPGRec_plus/utils/.
if os.path.isdir('CPGRec_plus') and not os.path.isdir('CPGRec_plus/utils'):
    shutil.rmtree('CPGRec_plus')
if not os.path.isdir('CPGRec_plus/utils'):
    !git clone --depth 1 https://github.com/HsipingLi/CPGRec-Plus.git CPGRec_plus
assert os.path.isdir('CPGRec_plus/utils'), "El git clone falló (revisá conexión/git). Reintentá esta celda."
os.makedirs('CPGRec_plus/data_exist', exist_ok=True)
print('Repo OK:', sorted(os.listdir('CPGRec_plus')))

Cloning into 'CPGRec_plus'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 39 (delta 1), reused 38 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 65.24 KiB | 337.00 KiB/s, done.
Resolving deltas: 100% (1/1), done.
Repo OK: ['.git', 'LICENSE', 'README.md', 'data_exist', 'main.py', 'models', 'requirements.txt', 'run.sh', 'utils']


In [ ]:
%%writefile CPGRec_plus/utils/graph_builders.py
"""
Builders inverted-index de los grafos juego-juego de CPGRec (estrictos SGC + laxo CNA).

Versión self-contained dentro del repo adaptado (espejo de `scripts/cpgrec_graphs.py`, que
tiene los tests de correctitud vs la referencia O(N^2)). Reemplaza el doble loop O(N^2) del
`build_edge_and`/`build_edge_or` original, inviable con ~22.676 juegos.

mappings: dict {game_idx: iterable de value_id}. Devuelve (src, dst) listas de ints con
aristas en ambas direcciones (grafo no dirigido), listas para torch.tensor(...).
"""
from __future__ import annotations
from collections import defaultdict
from typing import Dict, Iterable, List, Optional, Tuple


def _to_set_mapping(mapping: Dict[int, Iterable]) -> Dict[int, set]:
    out = {}
    for game, vals in mapping.items():
        s = set(vals)
        if s:
            out[game] = s
    return out


def _inverted_index(mapping: Dict[int, set]) -> Dict[object, List[int]]:
    inv: Dict[object, List[int]] = defaultdict(list)
    for game, vals in mapping.items():
        for v in vals:
            inv[v].append(game)
    return inv


def _pairs_to_src_dst(pairs: Iterable[Tuple[int, int]]) -> Tuple[List[int], List[int]]:
    src: List[int] = []
    dst: List[int] = []
    for a, b in pairs:
        src.append(a); dst.append(b)
        src.append(b); dst.append(a)
    return src, dst


def build_edge_and(mapping_A: Dict[int, Iterable],
                   mapping_B: Dict[int, Iterable]) -> Tuple[List[int], List[int]]:
    """Conecta i,j si comparten >=1 valor en A y >=1 valor en B (conexión ESTRICTA)."""
    A = _to_set_mapping(dict(mapping_A))
    B = _to_set_mapping(dict(mapping_B))
    common = A.keys() & B.keys()
    if not common:
        return [], []
    A = {g: A[g] for g in common}
    B = {g: B[g] for g in common}
    inv_A = _inverted_index(A)
    inv_B = _inverted_index(B)
    max_grp_A = max((len(v) for v in inv_A.values()), default=0)
    max_grp_B = max((len(v) for v in inv_B.values()), default=0)
    if max_grp_B <= max_grp_A:
        block_inv, other_map = inv_B, A
    else:
        block_inv, other_map = inv_A, B
    pairs = set()
    for group in block_inv.values():
        if len(group) < 2:
            continue
        local_inv: Dict[object, List[int]] = defaultdict(list)
        for game in group:
            for v in other_map[game]:
                local_inv[v].append(game)
        for sub in local_inv.values():
            if len(sub) < 2:
                continue
            for x in range(len(sub) - 1):
                gi = sub[x]
                for y in range(x + 1, len(sub)):
                    gj = sub[y]
                    pairs.add((gi, gj) if gi < gj else (gj, gi))
    return _pairs_to_src_dst(pairs)


def build_edge_or(*mappings: Dict[int, Iterable],
                  max_neighbors: Optional[int] = None,
                  seed: int = 42,
                  weight_mode: str = "size") -> Tuple[List[int], List[int]]:
    """Conecta i,j si comparten >=1 valor en CUALQUIER eje (LAXO). max_neighbors: tope de grado.

    Con tope, muestrea vecinos por nodo desde los grupos del inverted index (ponderado por
    tamaño) SIN materializar el producto cuadrático de los grupos gigantes (p.ej. 'Indie').
    Sin tope = exacto (solo viable en catálogos chicos).

    weight_mode controla el sesgo del muestreo de grupos (SOLO aplica con tope):
      "size"    = prob. proporcional al tamaño del grupo (original; sesga a hubs Indie/Action).
      "uniform" = cada eje/grupo del juego con igual prob. (C1a: refleja el perfil completo).
      "inverse" = prob. proporcional a 1/tamaño (C1b: favorece géneros raros / cola larga).
    """
    set_maps = [_to_set_mapping(dict(m)) for m in mappings]

    if max_neighbors is None:
        pairs = set()
        for m in set_maps:
            inv = _inverted_index(m)
            for group in inv.values():
                if len(group) < 2:
                    continue
                for idx in range(len(group) - 1):
                    gi = group[idx]
                    for jdx in range(idx + 1, len(group)):
                        gj = group[jdx]
                        pairs.add((gi, gj) if gi < gj else (gj, gi))
        return _pairs_to_src_dst(pairs)

    import random
    rng = random.Random(seed)
    invs = [_inverted_index(m) for m in set_maps]
    all_nodes = set()
    for m in set_maps:
        all_nodes |= m.keys()
    sampled = set()
    for u in all_nodes:
        pool = []
        for a, m in enumerate(set_maps):
            if u in m:
                for v in m[u]:
                    grp = invs[a][v]
                    if len(grp) >= 2:
                        pool.append(grp)
        if not pool:
            continue
        if weight_mode == "size":
            weights = [len(g) for g in pool]          # original: sesga a hubs (Indie/Action)
        elif weight_mode == "uniform":
            weights = [1.0 for _ in pool]             # C1a: cada eje cuenta igual
        elif weight_mode == "inverse":
            weights = [1.0 / len(g) for g in pool]    # C1b: favorece géneros raros
        else:
            raise ValueError(f"weight_mode invalido: {weight_mode}")
        chosen = set()
        attempts = 0
        max_attempts = max_neighbors * 20 + 10
        while len(chosen) < max_neighbors and attempts < max_attempts:
            grp = rng.choices(pool, weights=weights, k=1)[0]
            w = grp[rng.randrange(len(grp))]
            if w != u:
                chosen.add(w)
            attempts += 1
        for w in chosen:
            sampled.add((u, w) if u < w else (w, u))
    return _pairs_to_src_dst(sampled)


Writing CPGRec_plus/utils/graph_builders.py


In [ ]:
%%writefile CPGRec_plus/utils/dataloader_item.py
import os
import sys
from dgl.data.utils import save_graphs
from tqdm import tqdm
from scipy import stats
from utils.NegativeSampler import NegativeSampler
import pdb
import torch
import logging
logging.basicConfig(stream = sys.stdout, level = logging.INFO)
import numpy as np
import dgl
from dgl.data import DGLDataset
import pandas as pd
from sklearn import preprocessing
import pickle

# ADAPTADO: builders inverted-index (reemplazan el O(N^2) original). Ver graph_builders.py
from utils.graph_builders import build_edge_and as _build_edge_and
from utils.graph_builders import build_edge_or as _build_edge_or


class Dataloader_item_graph(DGLDataset):
    def __init__(self, dataloader_steam, or_cap=None, or_weight="size", data_exist="./data_exist"):
        """
        ADAPTADO para nuestro dataset:
          * usa builders inverted-index (evita el O(N^2) sobre 22.676 juegos).
          * fija num_nodes('game')=n_games en TODOS los grafos para que se alineen con
            item_embedding (juegos sin aristas estrictas igual son nodos aislados).
          * `or_cap`: tope de grado del grafo laxo G^Co (None = sin tope). Mitiga la
            explosión de aristas del eje 'genero'.
          * guarda y carga graph_or correctamente (el original tenía un bug).
        """
        self.publisher = dataloader_steam.publisher_mapping
        self.developer = dataloader_steam.developer_mapping
        self.genre = dataloader_steam.genre_mapping
        self.or_cap = or_cap
        self.or_weight = or_weight
        # nº total de juegos (del mapping de app_id), no del max índice con aristas
        self.n_games = len(dataloader_steam.app_id_mapping)

        os.makedirs(data_exist, exist_ok=True)
        path_dic_genre = data_exist + "/dic_genre.pkl"
        path_dic_pub = data_exist + "/dic_publisher.pkl"
        path_dic_dev = data_exist + "/dic_developer.pkl"

        if not os.path.exists(path_dic_genre) or not os.path.exists(path_dic_pub) or not os.path.exists(path_dic_dev):
            with open(path_dic_genre, 'wb') as f:
                pickle.dump(self.genre, f)
            with open(path_dic_pub, 'wb') as f:
                pickle.dump(self.publisher, f)
            with open(path_dic_dev, 'wb') as f:
                pickle.dump(self.developer, f)

        path_graph_and = data_exist + '/graph_and.bin'
        path_graph_or = data_exist + '/graph_or.bin'

        num_nodes_dict = {'game': self.n_games}

        '''graph 1: estrictos (SGC)'''
        if os.path.exists(path_graph_and):
            self.graph_and, _ = dgl.load_graphs(path_graph_and)
            self.graph_and = self.graph_and[0]
        else:
            logging.info("building strict game graphs (SGC) via inverted index...")
            self.genre_pub = self.build_edge_and(self.genre, self.publisher)
            self.genre_dev = self.build_edge_and(self.genre, self.developer)
            self.dev_pub = self.build_edge_and(self.developer, self.publisher)

            graph_data_and = {
                ('game', 'co_genre_pub', 'game'): self.genre_pub,
                ('game', 'co_genre_dev', 'game'): self.genre_dev,
                ('game', 'co_dev_pub', 'game'): self.dev_pub
            }
            self.graph_and = dgl.heterograph(graph_data_and, num_nodes_dict=num_nodes_dict)
            dgl.save_graphs(path_graph_and, [self.graph_and])

        '''graph 2: laxo (CNA / G^Co)'''
        if os.path.exists(path_graph_or):
            self.graph_or, _ = dgl.load_graphs(path_graph_or)
            self.graph_or = self.graph_or[0]
        else:
            logging.info(f"building lax game graph (CNA / G^Co) via inverted index "
                         f"(or_cap={self.or_cap}, or_weight={self.or_weight})...")
            self.genre_dev_pub = self.build_edge_or(self.genre, self.developer, self.publisher)
            graph_data_or = {
                ('game', 'co_or', 'game'): self.genre_dev_pub
            }
            self.graph_or = dgl.heterograph(graph_data_or, num_nodes_dict=num_nodes_dict)
            dgl.save_graphs(path_graph_or, [self.graph_or])

    def build_edge_and(self, mapping_A, mapping_B):
        src, dst = _build_edge_and(mapping_A, mapping_B)
        return (torch.tensor(src, dtype=torch.long), torch.tensor(dst, dtype=torch.long))

    def build_edge_or(self, mapping_genre, mapping_dev, mapping_pub):
        src, dst = _build_edge_or(mapping_genre, mapping_dev, mapping_pub,
                                  max_neighbors=self.or_cap, weight_mode=self.or_weight)
        return (torch.tensor(src, dtype=torch.long), torch.tensor(dst, dtype=torch.long))

    def __getitem__(self, i):
        pass

    def __len__(self):
        pass


Overwriting CPGRec_plus/utils/dataloader_item.py


In [ ]:
%%writefile CPGRec_plus/utils/dataloader_steam.py
import os
import sys
from dgl.data.utils import save_graphs
from tqdm import tqdm
from scipy import stats
from utils.NegativeSampler import NegativeSampler
import pdb
import torch
import logging
logging.basicConfig(stream=sys.stdout, level=logging.INFO)
import numpy as np
import dgl
from dgl.data import DGLDataset
import pandas as pd
from sklearn import preprocessing
import pickle
import dgl.function as fn
import pandas as pd



class Dataloader_steam_filtered(DGLDataset):
    def __init__(self, path, device = 'cpu'):


        self.device = device
        self.path = path
        self.user_id_path = self.path+"/users.txt"
        self.app_info_path = self.path+"/App_ID_Info.txt"
        self.app_id_path = self.path+"/app_id.txt"
        self.friends_path = self.path+"/friends.txt"
        self.genre_path = self.path+"/Games_Genres.txt"
        self.developer_path = self.path+"/Games_Developers.txt"
        self.publisher_path = self.path+"/Games_Publishers.txt"
        self.train_game_path = self.path+"/train_game.txt"
        self.valid_game_path = self.path+"/valid_data/valid_game.txt"
        self.test_game_path = self.path+"/test_data/test_game.txt"
        self.train_time_path = self.path+"/train_time.txt"
        self.graph_path = self.path + "/graph.bin"

        self.user_id_mapping = self.read_user_id_mapping(self.user_id_path)
        self.app_id_mapping = self.read_app_id_mapping(self.app_id_path)
        # ADAPTADO: nº de juegos parametrizado (el original tenía range(2675) hardcodeado)
        self.n_games = len(self.app_id_mapping)




        '''build valid and test data'''
        logging.info("build train data:")
        self.train_data = self.build_train_data(self.train_game_path)

        logging.info("build valid data:")
        self.valid_data = self.build_valid_data(self.valid_game_path)
        logging.info("build test data:")
        self.test_data = self.build_test_data(self.test_game_path)

        logging.info("read app info:")
        path_dic = "./data_exist/dic_app_info.pkl"
        self.dic_app_info = self.read_app_info(self.app_info_path, path_dic)
        self.dic_app_info_raw = self.read_app_info(self.app_info_path,
                                                        path_dic.replace('.pkl', '_raw.pkl'),
                                                        norm=False)


        self.process()
        dgl.save_graphs(self.graph_path, self.graph)




    def read_user_id_mapping(self, path):
        mapping = {}
        path_user_id_mapping = "./data_exist/user_id_mapping.pkl"
        if os.path.exists(path_user_id_mapping):
            with open(path_user_id_mapping, 'rb') as f:
                mapping = pickle.load(f)

        else:
            count = int(0)
            with open(path,'r') as f:
                lines = f.readlines()
                for line in lines:
                    line = line.strip()
                    if line not in mapping.keys():
                        mapping[line] = int(count)
                        count += 1
            with open(path_user_id_mapping, 'wb') as f:
                pickle.dump(mapping, f)
        return mapping



    def read_app_id_mapping(self, path):
        mapping = {}
        path_app_id_mapping = "./data_exist/app_id_mapping.pkl"
        if os.path.exists(path_app_id_mapping):
            with open(path_app_id_mapping, 'rb') as f:
                mapping = pickle.load(f)

        else:
            count = int(0)
            with open(path,'r') as f:
                lines = f.readlines()
                for line in lines:
                    line = line.strip()
                    if line not in mapping.keys():
                        mapping[line] = int(count)
                        count += 1
            with open(path_app_id_mapping, 'wb') as f:
                pickle.dump(mapping, f)
        return mapping


    def build_train_data(self, path):
        intr = {}
        path_valid_data = "./data_exist/train_data.pkl"
        if os.path.exists(path_valid_data):
            with open(path_valid_data, 'rb') as f:
                intr = pickle.load(f)
        else:
            with open(path, 'r') as f:
                lines = f.readlines()
                for line in tqdm(lines):
                    line = line.strip().split(',')
                    user = self.user_id_mapping[line[0]]
                    if user not in intr:
                        intr[user] = [self.app_id_mapping[game] for game in line[1:]]
            with open(path_valid_data, 'wb') as f:
                pickle.dump(intr, f)
        return intr




    def build_valid_data(self, path):
        intr = {}
        path_valid_data = "./data_exist/valid_data.pkl"
        if os.path.exists(path_valid_data):
            with open(path_valid_data, 'rb') as f:
                intr = pickle.load(f)
        else:
            with open(path, 'r') as f:
                lines = f.readlines()
                for line in tqdm(lines):
                    line = line.strip().split(',')
                    user = self.user_id_mapping[line[0]]
                    if user not in intr:
                        intr[user] = [self.app_id_mapping[game] for game in line[1:]]
            with open(path_valid_data, 'wb') as f:
                pickle.dump(intr, f)
        return intr



    def build_test_data(self, path):
        intr = {}
        path_valid_data = "./data_exist/test_data.pkl"
        if os.path.exists(path_valid_data):
            with open(path_valid_data, 'rb') as f:
                intr = pickle.load(f)
        else:
            with open(path, 'r') as f:
                lines = f.readlines()
                for line in tqdm(lines):
                    line = line.strip().split(',')
                    user = self.user_id_mapping[line[0]]
                    if user not in intr:
                        intr[user] = [self.app_id_mapping[game] for game in line[1:]]
            with open(path_valid_data, 'wb') as f:
                pickle.dump(intr, f)
        return intr









    def read_game_genre_mapping(self, path):
        mapping = {}
        path_game_type_mapping = "./data_exist/game_genre_mapping.pkl"
        if os.path.exists(path_game_type_mapping):
            with open(path_game_type_mapping, 'rb') as f:
                mapping = pickle.load(f)

            return mapping

        else:
            mapping_value2id = {}
            count = 0

            with open(path, 'r') as f:
                lines = f.readlines()
                for line in tqdm(lines):
                    line = line.strip().split(',')

                    if len(line)>=2 and line[1]!= '' and line[1] not in mapping_value2id:
                        mapping_value2id[line[1]] = count
                        count += 1

                for line in tqdm(lines):
                    line = line.strip().split(',')
                    if self.app_id_mapping[line[0]] not in mapping.keys() and line[1] != '':
                        mapping[self.app_id_mapping[line[0]]] = [line[1]]
                    elif self.app_id_mapping[line[0]] in mapping.keys() and line[1] != '':
                        mapping[self.app_id_mapping[line[0]]].append(line[1])


                for key in tqdm(mapping):
                    mapping[key] = [mapping_value2id[x] for x in mapping[key]]

                mapping_sort = {}
                for key in range(self.n_games):
                    if key not in mapping.keys():
                        mapping_sort[key] = []
                    else:
                        mapping_sort[key] = mapping[key]

                with open(path_game_type_mapping, 'wb') as f:
                    pickle.dump(mapping_sort, f)


            return mapping



    def read_game_dev_mapping(self, path):
        mapping = {}
        path_game_type_mapping = "./data_exist/game_dev_mapping.pkl"
        if os.path.exists(path_game_type_mapping):
            with open(path_game_type_mapping, 'rb') as f:
                mapping = pickle.load(f)

            return mapping

        else:
            mapping_value2id = {}
            count = 0

            with open(path, 'r') as f:
                lines = f.readlines()
                for line in tqdm(lines):
                    line = line.strip().split(',')

                    if len(line)>=2 and line[1]!= '' and line[1] not in mapping_value2id:
                        mapping_value2id[line[1]] = count
                        count += 1

                for line in tqdm(lines):
                    line = line.strip().split(',')
                    if self.app_id_mapping[line[0]] not in mapping.keys() and line[1] != '':
                        mapping[self.app_id_mapping[line[0]]] = [line[1]]
                    elif self.app_id_mapping[line[0]] in mapping.keys() and line[1] != '':
                        mapping[self.app_id_mapping[line[0]]].append(line[1])


                for key in tqdm(mapping):
                    mapping[key] = [mapping_value2id[x] for x in mapping[key]]

                mapping_sort = {}
                for key in range(self.n_games):
                    if key not in mapping.keys():
                        mapping_sort[key] = []
                    else:
                        mapping_sort[key] = mapping[key]

                with open(path_game_type_mapping, 'wb') as f:
                    pickle.dump(mapping_sort, f)


            return mapping_sort





    def read_game_pub_mapping(self, path):
        mapping = {}
        path_game_type_mapping = "./data_exist/game_pub_mapping.pkl"
        if os.path.exists(path_game_type_mapping):
            with open(path_game_type_mapping, 'rb') as f:
                mapping = pickle.load(f)

            return mapping

        else:
            mapping_value2id = {}
            count = 0

            with open(path, 'r') as f:
                lines = f.readlines()
                for line in tqdm(lines):
                    line = line.strip().split(',')

                    if len(line)>=2 and line[1]!= '' and line[1] not in mapping_value2id:
                        mapping_value2id[line[1]] = count
                        count += 1

                for line in tqdm(lines):
                    line = line.strip().split(',')
                    if self.app_id_mapping[line[0]] not in mapping.keys() and line[1] != '':
                        mapping[self.app_id_mapping[line[0]]] = [line[1]]
                    elif self.app_id_mapping[line[0]] in mapping.keys() and line[1] != '':
                        mapping[self.app_id_mapping[line[0]]].append(line[1])


                for key in tqdm(mapping):
                    mapping[key] = [mapping_value2id[x] for x in mapping[key]]

                mapping_sort = {}
                for key in range(self.n_games):
                    if key not in mapping.keys():
                        mapping_sort[key] = []
                    else:
                        mapping_sort[key] = mapping[key]

                with open(path_game_type_mapping, 'wb') as f:
                    pickle.dump(mapping_sort, f)


            return mapping_sort




    def read_play_time_rank(self, game_path, time_path):
        path = "./data_exist"
        path_tensor = path+"/tensor_user_game.pth"
        path_dic = path+"/dic_user_game.pkl"

        if os.path.exists(path_tensor) and os.path.exists(path_dic):
            tensor_user_game = torch.load(path_tensor)
            with open(path_dic,"rb") as f:
                dic_user_game = pickle.load(f)
            return tensor_user_game, dic_user_game

        else:
            ls = []
            dic_game = {}
            with open(game_path, 'r') as f_game:
                with open(time_path, 'r') as f_time:
                    lines_game = f_game.readlines()
                    lines_time = f_time.readlines()
                    for i in tqdm(range(len(lines_game))):
                        line_user_game = lines_game[i].strip().split(',')
                        user = self.user_id_mapping[line_user_game[0]]
                        line_game = line_user_game[1:]

                        line_time = lines_time[i].strip().split(',')[1:]
                        idx_time_filtered = [i for i in range(len(line_time)) if line_time[i] != r"\N"]
                        line_time_filtered = [float(line_time[i]) for i in idx_time_filtered]

                        if len(line_time_filtered) >=0:
                            dic_game[user] = []
                            ar_time = np.array(line_time_filtered)
                            time_mean = np.mean(ar_time)
                        else:
                            continue


                        for j in range(len(line_game)):
                            game = self.app_id_mapping[line_game[j]]
                            dic_game[user].append(game)
                            time = line_time[j]
                            if time == r'\N':
                                ls.append([user, game, time_mean])
                            else:
                                ls.append([user, game, float(time)])

                    with open(path_dic, 'wb') as f:
                        pickle.dump(dic_game, f)


            tensor = torch.tensor(ls)
            torch.save(tensor, path_tensor)
            return tensor, dic_game





    def game_genre_inter(self, mapping):
        game_type_inter = []
        path_game_genre_inter = "./data_exist/game_genre_inter.pkl"
        if os.path.exists(path_game_genre_inter):
            with open(path_game_genre_inter, 'rb') as f:
                game_type_inter = pickle.load(f)
        else:
            for key in tqdm(list(mapping.keys())):
                for type_key in mapping[key]:
                    game_type_inter.append([key,type_key])

            with open(path_game_genre_inter, 'wb') as f:
                pickle.dump(game_type_inter, f)

        return game_type_inter


    def game_dev_inter(self, mapping):
        game_type_inter = []
        path_game_genre_inter = "./data_exist/game_dev_inter.pkl"
        if os.path.exists(path_game_genre_inter):
            with open(path_game_genre_inter, 'rb') as f:
                game_type_inter = pickle.load(f)
        else:
            for key in tqdm(list(mapping.keys())):
                for type_key in mapping[key]:
                    game_type_inter.append([key,type_key])

            with open(path_game_genre_inter, 'wb') as f:
                pickle.dump(game_type_inter, f)

        return game_type_inter



    def game_pub_inter(self, mapping):
        game_type_inter = []
        path_game_genre_inter = "./data_exist/game_pub_inter.pkl"
        if os.path.exists(path_game_genre_inter):
            with open(path_game_genre_inter, 'rb') as f:
                game_type_inter = pickle.load(f)
        else:
            for key in tqdm(list(mapping.keys())):
                for type_key in mapping[key]:
                    game_type_inter.append([key,type_key])

            with open(path_game_genre_inter, 'wb') as f:
                pickle.dump(game_type_inter, f)

        return game_type_inter




    def read_app_info(self, path, path_dic, norm=True):

        if os.path.exists(path_dic):
            with open(path_dic, 'rb') as f:
                dic = pickle.load(f)
            return dic
        else:
            df = pd.read_csv(path, header=None)
            games = np.array(list(df.iloc[:,0])).reshape(-1,1)

            names = np.array(list(df.iloc[:,1])).reshape(-1,1)

            prices = np.array(list(df.iloc[:,3]))
            if norm: prices /= prices.max()
            prices_mean = prices.mean()
            prices = prices.reshape(-1,1)


            dates = df.iloc[:,4]
            if norm:
                dates = pd.to_datetime(dates).to_numpy().astype('int64')
                dates_mean = dates.mean()
            else:
                dates = pd.to_datetime(dates)
                dates_timestamps = dates.view(np.int64)
                dates_mean_timestamp = dates_timestamps.mean()
                dates_mean = pd.to_datetime(dates_mean_timestamp)



            if norm: dates = (dates.astype(float)/dates.max()).reshape(-1,1)
            else:
                dates = np.array(list(dates)).reshape(-1,1).astype(str)


            ratings = df.iloc[:,-3].replace(-1,np.nan)
            ratings_mean = ratings.mean()
            ratings = ratings.fillna(ratings_mean)
            ratings = np.array(ratings).reshape(-1,1)
            if norm: ratings = (ratings - ratings_mean)/np.std(ratings)

            app_info = np.hstack((names,prices,dates,ratings))
            dic = {}
            for i in range(len(games)):
                dic[self.app_id_mapping[str(games[i][0])]] = app_info[i]

            for game in self.app_id_mapping.keys():
                if int(game) not in games:
                    if norm: dic[self.app_id_mapping[game]] = np.array(["null", prices_mean, dates_mean, ratings_mean/100])
                    else: dic[self.app_id_mapping[game]] = np.array(["null", prices_mean, dates_mean, ratings_mean])
            with open(path_dic,'wb') as f:
                pickle.dump(dic, f)
            return dic







    def process(self):
        logging.info("reading genre,developer,publisher info...")
        self.genre_mapping = self.read_game_genre_mapping(self.genre_path)
        self.genre = self.game_genre_inter(self.genre_mapping)
        self.developer_mapping = self.read_game_dev_mapping(self.developer_path)
        self.developer = self.game_dev_inter(self.developer_mapping)
        self.publisher_mapping = self.read_game_pub_mapping(self.publisher_path)
        self.publisher = self.game_pub_inter(self.publisher_mapping)



        logging.info("reading user item play time...")
        self.user_game, self.dic_user_game = self.read_play_time_rank(self.train_game_path, self.train_time_path)



        if os.path.exists("./data_exist/graph.bin"):
            graph,_ = dgl.load_graphs("./data_exist/graph.bin")
            graph = graph[0]
            self.graph = graph

        else:
            graph_data = {


                ('game', 'developed by', 'developer'): (torch.tensor(self.developer)[:,0], torch.tensor(self.developer)[:,1]),

                ('developer', 'develop', 'game'): (torch.tensor(self.developer)[:,1], torch.tensor(self.developer)[:,0]),

                ('game', 'published by', 'publisher'):(torch.tensor(self.publisher)[:,0], torch.tensor(self.publisher)[:,1]),

                ('publisher', 'publish', 'game'): (torch.tensor(self.publisher)[:,1], torch.tensor(self.publisher)[:,0]),

                ('game', 'genre', 'type'): (torch.tensor(self.genre)[:,0], torch.tensor(self.genre)[:,1]),

                ('type', 'genred', 'game'): (torch.tensor(self.genre)[:,1], torch.tensor(self.genre)[:,0]),

                ('user', 'play', 'game'): (self.user_game[:, 0].long(), self.user_game[:, 1].long()),

                ('game', 'played by', 'user'): (self.user_game[:, 1].long(), self.user_game[:, 0].long())
            }
            # ADAPTADO: fijar nº de nodos game/user para que TODOS los juegos/usuarios sean
            # nodos (aunque algún juego no tenga interacciones o categoría) y el grafo
            # bipartito quede alineado con item_embedding y con los grafos de categoría.
            dev_t = torch.tensor(self.developer)
            pub_t = torch.tensor(self.publisher)
            gen_t = torch.tensor(self.genre)
            num_nodes_dict = {
                'game': self.n_games,
                'user': len(self.user_id_mapping),
                'developer': int(dev_t[:, 1].max()) + 1,
                'publisher': int(pub_t[:, 1].max()) + 1,
                'type': int(gen_t[:, 1].max()) + 1,
            }
            graph = dgl.heterograph(graph_data, num_nodes_dict=num_nodes_dict)

            self.graph = graph
            dgl.save_graphs("./data_exist/graph.bin",[graph])



    def __getitem__(self, i):
        pass

    def __len__(self):
        pass

Overwriting CPGRec_plus/utils/dataloader_steam.py


In [ ]:
%%writefile CPGRec_plus/utils/item_prompt_Qwen.py
"""
STUB adaptado del módulo PRG (item).

El original pegaba a la API de un LLM (Qwen2.5) para generar descripciones de cada juego y
luego las embebía con bge-m3. Aquí, en cambio, `get_item_embedding_with_LLM()` devuelve un
embedding externo PRECOMPUTADO que el driver escribe en disco:

  * CPGRec base / +PER : una matriz de ceros (n_games, d) -> la rama de contenido no aporta
    información (se neutraliza); el modelo funciona solo con los módulos de grafo.
  * +PRG-SBERT (NICE)  : nuestros embeddings SBERT de las descripciones, alineados al índice
    de juego del dataloader -> aproximación del módulo PRG sin LLM (se reporta como ablación).

Ruta del archivo: variable de entorno ITEM_EMB_EXTERNAL, o ./data_exist/item_emb_external.pth
(tensor float32 (n_games, d)). La firma se mantiene para no tocar models/model.py.
"""
import os
import torch


def get_item_embedding_with_LLM(*args, **kwargs):
    path = os.environ.get("ITEM_EMB_EXTERNAL", "./data_exist/item_emb_external.pth")
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"No encontré {path}. El driver (run_cpgrec.py) debe escribir el embedding "
            f"externo de items ANTES de construir el modelo (ceros para base, SBERT para "
            f"+PRG-SBERT)."
        )
    emb = torch.load(path)
    return emb.float()


Overwriting CPGRec_plus/utils/item_prompt_Qwen.py


In [ ]:
%%writefile CPGRec_plus/utils/GetWeight_PENR.py
"""
PENR (Popularity-guided Edges and Nodes Reweighting) — ADAPTADO.

El original tenía los cortes hot/cold HARDCODEADOS a la distribución de grados del dataset
del paper (quantile_low=50000, high=850000 para aristas; 570/33000 para nodos). Eso no
aplica a nuestro catálogo. Aquí los cortes se CALCULAN como los cuantiles 0.2 / 0.8 del
grado real (np.quantile, robusto a tensores grandes donde torch.quantile falla).

theta1: peso extra a las aristas que salen de juegos populares (hot).
theta2: atenúa el peso de los nodos juego populares (hot).
theta3: realza el peso de los nodos juego de cola larga (cold).
"""
import dgl
import numpy as np
import torch


def _quantile(t: torch.Tensor, q: float) -> float:
    return float(np.quantile(t.detach().cpu().numpy(), q))


def get_penr_weight(theta1=80, theta2=0.5, theta3=3, data_exist="./data_exist"):
    torch.manual_seed(2023)
    graph = dgl.load_graphs(data_exist + "/graph.bin")[0][0]
    graph = dgl.edge_type_subgraph(graph, etypes=['played by'])

    # --- aristas: popularidad del juego origen de cada arista 'played by' ---
    outdeg_game = graph.out_degrees(
        graph.edges(etype=('game', 'played by', 'user'))[0], etype='played by').float()
    q_high_e = _quantile(outdeg_game, 0.80)
    weight_edge = torch.ones_like(outdeg_game)
    idx_high = outdeg_game > q_high_e
    weight_edge[idx_high] = weight_edge[idx_high] * theta1
    torch.save(idx_high, data_exist + "/idx_high.pth")
    torch.save(weight_edge, data_exist + "/weight_edge.pth")

    # --- nodos: grado de cada nodo-juego ---
    outdeg_game_2 = graph.out_degrees(graph.nodes(ntype='game'), etype='played by').float()
    q_low_n = _quantile(outdeg_game_2, 0.20)
    q_high_n = _quantile(outdeg_game_2, 0.80)
    weight_node = torch.ones_like(outdeg_game_2)
    idx_low = outdeg_game_2 <= q_low_n
    idx_high2 = outdeg_game_2 > q_high_n
    weight_node[idx_low] = weight_node[idx_low] * theta3
    weight_node[idx_high2] = weight_node[idx_high2] * theta2
    torch.save(torch.where(idx_low), data_exist + "/game_cold.pth")
    torch.save(torch.where(idx_high2), data_exist + "/game_hot.pth")
    torch.save(weight_node, data_exist + "/weight_node.pth")

    return weight_edge, weight_node


Overwriting CPGRec_plus/utils/GetWeight_PENR.py


In [ ]:
%%writefile CPGRec_plus/run_cpgrec.py
"""
Driver de entrenamiento de CPGRec / CPGRec+ — ADAPTADO a nuestro pipeline.

Reemplaza a main.py para nuestro uso. Diferencias clave:
  * fallback a CPU si no hay GPU; rutas via --path / --data_exist (sin os.chdir).
  * pasa --or_cap al builder del grafo laxo G^Co.
  * toggles de alcance:  --use_per {0,1}  (PER, reweighting con signo),
                         --use_llm {0,1}  (rama PRG; el embedding externo lo decide el
                                           driver: ceros=base, SBERT=+PRG-SBERT).
  * PER vectorizado (el original iteraba juego por juego sobre todas las interacciones).
  * al terminar exporta data_exist/final_embeddings.pth = {'user': e_u, 'game': e_i} para
    evaluar con NUESTRO harness (leave-one-out temporal) fuera de este script.

Uso (desde la carpeta del repo, con steam_data/ ya construido):
    python run_cpgrec.py --path ./steam_data --or_cap 50 --use_per 0 --use_llm 0 \
                         --epochs 300 --embed_size 32 --lr 0.03
"""
import argparse
import os
import pickle
import sys

import numpy as np
import torch
import torch.nn.functional as F
import dgl
import logging
logging.basicConfig(stream=sys.stdout, level=logging.INFO)

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from utils.dataloader_steam import Dataloader_steam_filtered
from utils.dataloader_item import Dataloader_item_graph
from models.model import Proposed_model
from models.Predictor import Predictor
from scipy.stats import boxcox, f


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument('--path', default='./steam_data', type=str)
    p.add_argument('--data_exist', default='./data_exist', type=str)
    p.add_argument('--seed', default=3407, type=int)
    p.add_argument('--embed_size', default=32, type=int)
    p.add_argument('--lr', default=0.03, type=float)
    p.add_argument('--epochs', default=300, type=int)
    p.add_argument('--m', default=4.0, type=float)               # NSR: reweighting de negativos
    p.add_argument('--gamma', default=80.0, type=float)          # fusion w_or/w_and/w_self
    p.add_argument('--layers_and', default=2, type=int)
    p.add_argument('--layers_or', default=4, type=int)
    p.add_argument('--layers_user_game', default=2, type=int)
    p.add_argument('--attention_and', default=1, type=int)
    p.add_argument('--param_decay', default=0.1, type=float)     # beta de CNA (eq.3)
    p.add_argument('--or_cap', default=50, type=int)             # tope de grado de G^Co (0=sin tope)
    p.add_argument('--or_weight', default='size', type=str)      # muestreo G^Co: size|uniform|inverse
    p.add_argument('--neg_mode', default='random', type=str)     # random | observed_aux (C2)
    p.add_argument('--neg_aux_weight', default=1.0, type=float)  # peso del termino de neg. observados
    p.add_argument('--use_per', default=0, type=int)
    p.add_argument('--freeze_user_llm', default=1, type=int)  # 1=zero+freeze (default); 0=entrenable (como el repo)
    p.add_argument('--use_llm', default=0, type=int)
    p.add_argument('--item_emb', default='', type=str)           # .pth (n_games,d) para +PRG-SBERT
    p.add_argument('--gpu', default=0, type=int)
    p.add_argument('--eval_every', default=50, type=int)
    args = p.parse_args()
    args.attention_and = bool(args.attention_and)
    return args


def setup_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    import random
    random.seed(seed)
    torch.backends.cudnn.deterministic = True


def construct_negative_graph(graph, etype, device):
    utype, _, vtype = etype
    src, _ = graph.edges(etype=etype)
    src = src.to(device)
    dst = torch.randint(graph.num_nodes(vtype), size=src.shape).to(device)
    return dst, dgl.heterograph(
        {etype: (src, dst)},
        num_nodes_dict={nt: graph.number_of_nodes(nt) for nt in graph.ntypes})


def build_observed_neg_graph(path, DataLoader, n_games, device):
    """Grafo (user->game) con interacciones is_recommended=False (negativos observados).
    Mapea ids crudos a indices del repo via los mappings del dataloader; robusto a faltantes."""
    fn = os.path.join(path, 'train_game_neg.txt')
    if not os.path.exists(fn):
        return None, 0
    umap = DataLoader.user_id_mapping; amap = DataLoader.app_id_mapping
    def _key(x, mp):
        if x in mp:
            return x
        if x.lstrip('-').isdigit() and int(x) in mp:
            return int(x)
        return None
    src, dst = [], []
    with open(fn) as f:
        for line in f:
            parts = line.strip().split(',')
            if not parts or parts[0] == '':
                continue
            uk = _key(parts[0], umap)
            if uk is None:
                continue
            ui = umap[uk]
            for a in parts[1:]:
                ak = _key(a, amap)
                if ak is not None:
                    src.append(ui); dst.append(amap[ak])
    if not src:
        return None, 0
    g = dgl.heterograph(
        {('user', 'play', 'game'): (torch.tensor(src), torch.tensor(dst))},
        num_nodes_dict={'user': len(umap), 'game': n_games}).to(device)
    return g, len(src)


def build_per_weight(DataLoader, n_games, device):
    """PER vectorizado. Devuelve weight_PER (len = nº interacciones) o None si no aplica."""
    try:
        inter = DataLoader.user_game.clone().float()           # [N,3] user,game,time
        dic = DataLoader.dic_app_info                          # idx -> [name,price,date,rating]
        ratings = np.array([float(dic[g][-1]) for g in range(n_games)], dtype=np.float64)
        gidx = inter[:, 1].long().cpu().numpy()
        rating_col = torch.tensor(ratings[gidx], dtype=torch.float32).unsqueeze(1)
        inter = torch.hstack([inter, rating_col])              # [N,4] +rating
        # nan -> media de columna
        means = torch.nanmean(inter, dim=0)
        inter = torch.where(torch.isnan(inter), means, inter)

        t, r = inter[:, 2], inter[:, 3]
        t, r = t - t.min(), r - r.min()
        t = torch.tensor(boxcox(t.cpu().numpy() + 1e-6)[0])
        r = torch.tensor(boxcox(r.cpu().numpy() + 1e-6)[0])
        t = (t - t.mean()) / t.std()
        r = (r - r.mean()) / r.std()
        f_stat = t ** 2 / r ** 2

        alpha = 0.4
        Q_u = f.ppf(1 - alpha, 1, 1)
        mask_sig = f_stat >= Q_u
        mask_like = (t > r) & mask_sig
        mask_hate = (t <= r) & mask_sig
        mask_mode = ~(mask_like | mask_hate)

        def norm_pdf(x):
            return (1 / torch.sqrt(2 * torch.tensor(np.pi))) * torch.exp(-x ** 2 / 2)
        info_cont = ((norm_pdf(t) * norm_pdf(r)) + 1e-6).log() * (-1)

        w = torch.ones_like(info_cont)
        w[mask_like] = info_cont[mask_like]
        w[mask_hate] = info_cont[mask_hate] * (-1)
        w[mask_mode] = 0.0
        return w.float().to(device)
    except Exception as e:
        logging.warning(f"[PER] no se pudo computar weight_PER ({e}); se entrena sin PER.")
        return None


def main():
    args = parse_args()
    setup_seed(args.seed)
    device = torch.device(f"cuda:{args.gpu}" if torch.cuda.is_available() else "cpu")
    logging.info(f"device = {device}")
    os.makedirs(args.data_exist, exist_ok=True)

    # --- datos + grafos ---
    DataLoader = Dataloader_steam_filtered(args.path, device=device)
    n_games = len(DataLoader.app_id_mapping)
    n_users = len(DataLoader.user_id_mapping)
    logging.info(f"n_users={n_users}  n_games={n_games}")

    DataLoader_item = Dataloader_item_graph(
        DataLoader, or_cap=(None if args.or_cap == 0 else args.or_cap),
        or_weight=args.or_weight, data_exist=args.data_exist)
    graph_item_and = DataLoader_item.graph_and
    graph_item_or = DataLoader_item.graph_or
    graph = dgl.edge_type_subgraph(
        DataLoader.graph, [('user', 'play', 'game'), ('game', 'played by', 'user')])
    if args.neg_mode == 'observed_aux':
        graph_neg_obs, n_obs = build_observed_neg_graph(args.path, DataLoader, n_games, device)
        logging.info(f"[C2] neg_mode=observed_aux: {n_obs} aristas neg observadas; aux_weight={args.neg_aux_weight}")
        if graph_neg_obs is None:
            logging.warning("[C2] sin negativos observados utilizables -> se entrena solo con BPR (=random).")
    else:
        graph_neg_obs = None

    # --- embedding externo de items (rama PRG): ceros (base) o SBERT (+PRG-SBERT) ---
    if args.use_llm and args.item_emb and os.path.exists(args.item_emb):
        item_emb_ext = torch.load(args.item_emb).float()
        assert item_emb_ext.shape[0] == n_games, \
            f"item_emb ({item_emb_ext.shape}) no coincide con n_games={n_games}"
        logging.info(f"[PRG] usando embeddings externos {tuple(item_emb_ext.shape)}")
    else:
        item_emb_ext = torch.zeros(n_games, args.embed_size)
        if args.use_llm:
            logging.warning("[PRG] use_llm=1 pero sin --item_emb válido; uso ceros (=base).")
    torch.save(item_emb_ext, os.path.join(args.data_exist, "item_emb_external.pth"))
    os.environ["ITEM_EMB_EXTERNAL"] = os.path.join(args.data_exist, "item_emb_external.pth")

    # --- PER ---
    weight_PER = build_per_weight(DataLoader, n_games, device) if args.use_per else None

    # --- modelo ---
    model = Proposed_model(args, graph, graph_item_and, graph_item_or, device, weight_PER)
    model = model.to(device)

    # Neutralizar la rama PRG cuando no se usa contenido (item ceros) y SIEMPRE la de
    # usuario (no aproximamos PRG de usuario): se congela para que no inyecte ruido.
    if args.freeze_user_llm:
        with torch.no_grad():
            model.user_embedding_LLM.zero_()
        model.user_embedding_LLM.requires_grad_(False)
    # freeze_user_llm=0: user_embedding_LLM (randn) queda ENTRENABLE, como en el repo oficial
    if not args.use_llm:
        with torch.no_grad():
            model.item_embedding_LLM.zero_()
        model.item_embedding_LLM.requires_grad_(False)

    predictor = Predictor()
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=args.lr)

    logging.info("entrenando...")
    best_loss = float('inf')
    best_state = None
    for epoch in range(args.epochs):
        model.train()
        dst, graph_neg = construct_negative_graph(graph, ('user', 'play', 'game'), device)
        h, h_and, h_or = model()
        score = predictor(graph, h, ('user', 'play', 'game'), device)
        score_neg = predictor(graph_neg, h, ('user', 'play', 'game'), device)
        score_neg_rw = score_neg * (score_neg.sigmoid() * args.m)
        loss = (-((score - score_neg_rw).sigmoid().clamp(min=1e-8, max=1 - 1e-8).log()[:, 0])).sum()
        if graph_neg_obs is not None:                       # C2: baja el score de neg. observados
            score_obs = predictor(graph_neg_obs, h, ('user', 'play', 'game'), device)
            loss = loss + args.neg_aux_weight * F.softplus(score_obs).sum()
        opt.zero_grad()
        loss.backward()
        opt.step()
        cur = float(loss)
        if cur < best_loss:                       # nuevo mínimo -> snapshot
            best_loss = cur
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if epoch % 10 == 0 or epoch == args.epochs - 1:
            logging.info(f"epoch {epoch:4d}  loss = {cur:.4f}  (best = {best_loss:.4f})")

    # --- exportar el MEJOR modelo (no el último) ---
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        h, _, _ = model()
    out = {
        'user': h['user'].detach().cpu().float(),
        'game': h['game'].detach().cpu().float(),
    }
    torch.save(out, os.path.join(args.data_exist, "final_embeddings.pth"))
    logging.info(f"guardado final_embeddings.pth (best loss = {best_loss:.4f})  "
                 f"user={tuple(out['user'].shape)} game={tuple(out['game'].shape)}")
    logging.info("listo.")


if __name__ == "__main__":
    main()


Writing CPGRec_plus/run_cpgrec.py


In [ ]:
# ====== Escribir steam_data/ desde el split 80/10/10 (formato que lee el driver) ======
import shutil, csv as _csv
STEAM_DATA = 'CPGRec_plus/steam_data'
DATA_EXIST = 'CPGRec_plus/data_exist'

_train = inter[inter['split'] == 0]; _valid = inter[inter['split'] == 1]; _test = inter[inter['split'] == 2]

# Invalida cache de data_exist/steam_data si cambió la config de datos.
os.makedirs(DATA_EXIST, exist_ok=True)
_marker = os.path.join(DATA_EXIST, '_data_marker.json')
_cur = {'tier': TIER, 'subsample': SUBSAMPLE_USERS, 'seed': SEED, 'split': '80/10/10'}
_old = json.load(open(_marker)) if os.path.exists(_marker) else None
if _old != _cur:
    shutil.rmtree(DATA_EXIST, ignore_errors=True); shutil.rmtree(STEAM_DATA, ignore_errors=True)
    os.makedirs(DATA_EXIST, exist_ok=True)
    print('Config de datos cambió -> cache data_exist/steam_data invalidado.')
json.dump(_cur, open(_marker, 'w'))
for _sub in ('', '/valid_data', '/test_data'):
    os.makedirs(STEAM_DATA + _sub, exist_ok=True)

def _wl(path, lines):
    with open(path, 'w', encoding='utf-8', newline='') as f:
        f.write('\n'.join(lines) + ('\n' if lines else ''))

users_all = sorted(int(u) for u in inter['user_id'].unique())
_wl(f'{STEAM_DATA}/app_id.txt', [str(a) for a in CATALOG])
_wl(f'{STEAM_DATA}/users.txt', [str(u) for u in users_all])

def _wcat(path, cmap):
    _wl(path, [f'{a},{v}' for a in CATALOG for v in cmap.get(a, [])])
_wcat(f'{STEAM_DATA}/Games_Genres.txt', genre_map)
_wcat(f'{STEAM_DATA}/Games_Developers.txt', dev_map)
_wcat(f'{STEAM_DATA}/Games_Publishers.txt', pub_map)

# App_ID_Info.txt: app_id,name,type,price,release_date,metascore,0,1 (8 col, formato nativo)
_catx = cat.set_index('app_id')
def _g(row, col, default):
    if row is None: return default
    v = row.get(col)
    return v if (v is not None and not (isinstance(v, float) and pd.isna(v))) else default
with open(f'{STEAM_DATA}/App_ID_Info.txt', 'w', encoding='utf-8', newline='') as f:
    w = _csv.writer(f)
    for a in CATALOG:
        row = _catx.loc[a] if a in _catx.index else None
        price = pd.to_numeric(_g(row, 'price', 0.0), errors='coerce'); price = float(price) if pd.notna(price) else 0.0
        meta = pd.to_numeric(_g(row, 'metascore', -1), errors='coerce'); meta = int(meta) if pd.notna(meta) else -1
        # Normaliza CUALQUIER fecha (Timestamp del parquet, str con/sin hora, NaT) a
        # 'YYYY-MM-DD' uniforme: el dataloader del repo hace pd.to_datetime(col) e INFIERE
        # un solo formato -> mezclar 'YYYY-MM-DD HH:MM:SS' (reales) con 'YYYY-MM-DD'
        # (default) lo revienta. Date-only = misma convencion que H2.
        _dt = pd.to_datetime(_g(row, 'release_date', None), errors='coerce')
        date = _dt.strftime('%Y-%m-%d') if pd.notna(_dt) else '2015-01-01'
        w.writerow([a, _g(row, 'name', 'null'), 'game', price, date, meta, 0, 1])

# train_game/train_time (adyacencia alineada) y valid/test (multi-juego, sin tiempos)
def _adj_time(df, gpath, tpath):
    grp = df.groupby('user_id'); gl, tl = [], []
    for u in users_all:
        if u in grp.groups:
            gg = grp.get_group(u)
            games_u = [str(int(x)) for x in gg['app_id']]; time_u = [f'{float(t):.4f}' for t in gg['playtime']]
        else:
            games_u, time_u = [], []
        gl.append(','.join([str(u)] + games_u)); tl.append(','.join([str(u)] + time_u))
    _wl(gpath, gl); _wl(tpath, tl)

def _adj(df, path):
    grp = df.groupby('user_id')
    _wl(path, [','.join([str(int(u))] + [str(int(x)) for x in grp.get_group(u)['app_id']]) for u in grp.groups])

_adj_time(_train, f'{STEAM_DATA}/train_game.txt', f'{STEAM_DATA}/train_time.txt')
_adj(_valid, f'{STEAM_DATA}/valid_data/valid_game.txt')
_adj(_test,  f'{STEAM_DATA}/test_data/test_game.txt')
print(f'steam_data escrito: {len(CATALOG):,} juegos, {len(users_all):,} usuarios, '
      f'train={len(_train):,} valid={len(_valid):,} test={len(_test):,} interac')

Config de datos cambió -> cache data_exist/steam_data invalidado.
steam_data escrito: 9,198 juegos, 62,944 usuarios, train=4,106,088 valid=513,261 test=513,261 interac


## 5. Entrenamiento multi-seed CORREGIDO — UNA sola corrida (1000 épocas + `user_embedding_LLM` entrenable)

Config corregida (validada en SCGRec por `diag_train_T1`: 1000 épocas ≫ 300, y descongelar
`user_embedding_LLM` ayuda): **`EPOCHS=1000`** y **`--freeze_user_llm 0`**, 3 variantes (base + +PER +
+PER+PRG) × 3 seeds, **todo en una corrida**.

**UCSD es el más liviano de los 3 datasets** (~60k usuarios tras 5-core) → los 9 entrenamientos
entran cómodos en **una sesión Pro+ SIN subsample** (`SUBSAMPLE_USERS=None`). Sanity rápido: poné
`TIER='T1'` (50 épocas).

> Correr: setup DGL (1ª celda) sola → reinicia → *Ejecutar todo desde arriba*. **No hay que descargar
> nada**: todo se imprime al final. Igual guarda `comparativo_ucsd_corregido/per_seed_partial.json`
> tras cada seed como seguro ante caídas. Vigilá en el log que aparezca `[PER]` computado y **NO** el
> warning `"[PER] ... se entrena sin PER"` (fallback).

In [ ]:
# (multi-seed) asegurar data_exist antes de escribir item_emb
os.makedirs('CPGRec_plus/data_exist', exist_ok=True)
# ====== (+PRG-SBERT) Embeddings SBERT de items, alineados al indice del repo (= orden de CATALOG) ======
# Variante propia: aproxima la rama PRG (LLM Qwen del paper) con SBERT liviano. UCSD trae tags ->
# texto = nombre + generos + developer + publisher + tags.
item_emb_arg = ''
if USE_LLM:
    import torch, numpy as _np
    from sentence_transformers import SentenceTransformer
    def _lst(v):
        if isinstance(v, _np.ndarray): return [x for x in v.tolist() if x is not None]
        if isinstance(v, list):        return [x for x in v if x is not None]
        return [] if v is None else [v]
    _catx = cat.set_index('app_id')
    def _txt(a):
        if a not in _catx.index: return 'unknown game'
        row = _catx.loc[a]
        nm = row.get('name')
        nm = str(nm) if isinstance(nm, str) and nm.strip() else 'unknown game'
        gen = ', '.join(map(str, _lst(row.get('genres'))))
        dev = ', '.join(map(str, _lst(row.get('developers'))))
        pub = ', '.join(map(str, _lst(row.get('publishers'))))
        tags = ', '.join(map(str, _lst(row.get('tags'))))
        parts = [nm]
        if gen: parts.append('Genres: ' + gen)
        if dev: parts.append('Developer: ' + dev)
        if pub: parts.append('Publisher: ' + pub)
        if tags: parts.append('Tags: ' + tags)
        return '. '.join(parts)
    texts = [_txt(a) for a in CATALOG]      # alineado a CATALOG = orden de app_id.txt = indice del repo
    _dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    _sbert = SentenceTransformer(EMB_MODEL, device=_dev)
    _emb = _sbert.encode(texts, batch_size=128, show_progress_bar=True,
                         convert_to_numpy=True, normalize_embeddings=True).astype('float32')
    assert _emb.shape[0] == len(CATALOG), (_emb.shape, len(CATALOG))
    # Liberar el modelo SBERT (RAM + GPU) antes de entrenar el GNN.
    import gc
    del _sbert
    if _dev == 'cuda': torch.cuda.empty_cache()
    gc.collect()
    # Reducir la dim del embedding de contenido: el modelo crea user_embedding_LLM de
    # (n_users x dim). PCA -> PRG_EMB_DIM evita el OOM (bajo riesgo en UCSD por ser chico).
    if PRG_EMB_DIM and _emb.shape[1] > PRG_EMB_DIM:
        from sklearn.decomposition import PCA
        _emb = PCA(n_components=PRG_EMB_DIM, random_state=SEED).fit_transform(_emb).astype('float32')
        _emb = _emb / (_np.linalg.norm(_emb, axis=1, keepdims=True) + 1e-9)
        print(f'PCA 384 -> {_emb.shape[1]} dims')
    torch.save(torch.tensor(_emb), 'CPGRec_plus/data_exist/item_emb_sbert.pth')
    item_emb_arg = '--item_emb ./data_exist/item_emb_sbert.pth'
    print('+PRG-SBERT: item_emb_sbert.pth', _emb.shape, '(modelo', EMB_MODEL + ')')
else:
    print('USE_LLM=0 -> CPGRec base (sin rama PRG; item_emb_arg vacio).')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/72 [00:00<?, ?it/s]

PCA 384 -> 64 dims
+PRG-SBERT: item_emb_sbert.pth (9198, 64) (modelo sentence-transformers/all-MiniLM-L6-v2)


In [ ]:
# ====== ALS (implicit) — techo colaborativo, confianza = 1 + alpha*playtime ======
import scipy.sparse as _sp
from implicit.als import AlternatingLeastSquares
als_users = sorted(int(u) for u in train['user_id'].unique())
_u2i = {u:i for i,u in enumerate(als_users)}
_a2i = {a:i for i,a in enumerate(CATALOG)}
_tr = train[train['app_id'].isin(_a2i)]
_rows = _tr['user_id'].map(_u2i).to_numpy()
_cols = _tr['app_id'].map(_a2i).to_numpy()
_conf = (1.0 + ALS_ALPHA * np.log1p(_tr['playtime'].fillna(0).clip(lower=0).to_numpy())).astype('float32')
_ui = _sp.csr_matrix((_conf, (_rows, _cols)), shape=(len(als_users), len(CATALOG)))
als = AlternatingLeastSquares(factors=ALS_FACTORS, regularization=ALS_REG, iterations=ALS_ITERS, random_state=SEED, use_gpu=False)
als.fit(_ui)
als_uf = np.asarray(als.user_factors); als_if = np.asarray(als.item_factors)
print('ALS listo:', als_uf.shape, als_if.shape)


/usr/local/lib/python3.12/dist-packages/implicit/gpu/__init__.py:28: UserWarning: Disabling GPU support because of 'libcublas.so.13: cannot open shared object file: No such file or directory'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

ALS listo: (62944, 64) (9198, 64)


### eval_set + recomendaciones de los modelos sin-seed (MostPop, ALS)

In [ ]:
# ====== eval_set (fijo) + helpers + recs de modelos SIN seed (MostPop, ALS) ======
if N_EVAL_ANALYSIS and len(eval_users) > N_EVAL_ANALYSIS:
    _rng = np.random.default_rng(SEED)
    eval_set = sorted(_rng.choice(np.array(eval_users), size=N_EVAL_ANALYSIS, replace=False).tolist())
else:
    eval_set = list(eval_users)
print(f'eval_set: {len(eval_set):,} usuarios | SEEDS={SEEDS}')

idx2app_als = {i:a for a,i in _a2i.items()}
def _topn(scores, idx2app, seen, topn):
    rec=[]
    for j in np.argsort(-scores):
        a=idx2app[j]
        if a not in seen:
            rec.append(a)
            if len(rec)>=topn: break
    return rec
def recs_emb(e_u, e_i, umap, idx2app, users, topn):
    out={}; nfb=0
    for u in users:
        seen=train_items_per_user.get(u,set()); key=str(u)
        if key not in umap:
            out[u]=[i for i in popular_list if i not in seen][:topn]; nfb+=1; continue
        out[u]=_topn(e_i @ e_u[umap[key]], idx2app, seen, topn)
    return out, nfb

recs_fixed = {}
recs_fixed['Most Popular'] = {u:[i for i in popular_list if i not in train_items_per_user.get(u,set())][:TOPN] for u in eval_set}
als_umap = {str(u):_u2i[u] for u in als_users}
recs_fixed['ALS'], _ = recs_emb(als_uf, als_if, als_umap, idx2app_als, eval_set, TOPN)
metrics_fixed = {m: paper_metrics(recs_fixed[m], test_items_per_user, cat_maps=CAT_MAPS) for m in recs_fixed}
print('recs fijas (no dependen del seed):', list(recs_fixed))


eval_set: 57,390 usuarios | SEEDS=[42, 1, 2]
recs fijas (no dependen del seed): ['Most Popular', 'ALS']


### Loop multi-seed (entrena base + +PER + +PER+PRG por cada seed; persiste parcial por seed)

In [ ]:
# ====== Loop multi-seed: entrena base + +PER + +PER+PRG por cada seed ======
# Las 3 variantes GNN se entrenan en la MISMA sesion -> deltas pareados validos (leccion del
# proyecto: entre corridas hay +-2-4% de ruido GPU, del orden del efecto). +PER y +PER+PRG son
# lo nuevo; base se RE-ENTRENA aqui para parear. +PRG-solo NO se re-corre (negativo ya 0/3 en
# CPGRec_comparativo_multiseed_T2_ejecutado).
import shutil as _sh, pickle as _pk, torch as _t, json as _json
def _bucket(n): return '2-5' if n<=5 else '6-20' if n<=20 else '21-50' if n<=50 else '51+'
def _u_nr(rec, rel, k):
    hits=[1 if it in rel else 0 for it in rec[:k]]; nh=sum(hits)
    dcg=sum(1/math.log2(i+2) for i,h in enumerate(hits) if h)
    idcg=sum(1/math.log2(i+2) for i in range(min(len(rel),k)))
    return (dcg/idcg if idcg>0 else 0.0, nh/len(rel) if rel else 0.0)
buckets=['2-5','6-20','21-50','51+']
_act={u:_bucket(len(train_items_per_user.get(u,set()))) for u in eval_set}
def _longtail(recs_u):
    out={}
    for b in buckets:
        us=[u for u in eval_set if _act[u]==b and test_items_per_user.get(u)]
        if not us: continue
        out[b]={'n':len(us),
                'NDCG@10':float(np.mean([_u_nr(recs_u[u],test_items_per_user[u],10)[0] for u in us])),
                'Recall@10':float(np.mean([_u_nr(recs_u[u],test_items_per_user[u],10)[1] for u in us]))}
    return out
def _common(flags):
    return ("cd CPGRec_plus && python run_cpgrec.py --path ./steam_data --data_exist ./data_exist "
            f"{flags} --freeze_user_llm {FREEZE_USER_LLM} --or_cap {OR_CAP} --or_weight {OR_WEIGHT} --neg_mode random --neg_aux_weight 1 "
            f"--epochs {EPOCHS} --embed_size {EMBED_SIZE} --lr {LR} --m {M_NSR} --gamma 80.0 "
            f"--param_decay {PARAM_DECAY}")
def _clean(d):
    return {k:(float(v) if isinstance(v,(int,float,np.floating)) else v) for k,v in d.items()}

# (nombre del modelo, flags de alcance). item_emb_arg viene de la celda SBERT (USE_LLM=1).
GNN_VARIANTS=[('CPGRec base',     '--use_per 0 --use_llm 0'),
              ('CPGRec +PER',     '--use_per 1 --use_llm 0'),
              ('CPGRec +PER+PRG', f'--use_per 1 --use_llm 1 {item_emb_arg}')]
GNN_MODELS=[m for m,_ in GNN_VARIANTS]

per_seed=[]; seed0_recs={}
app_id_mapping=None; user_id_mapping=None; idx2app_cpg=None
os.makedirs('comparativo_ucsd_corregido', exist_ok=True); _PARTIAL='comparativo_ucsd_corregido/per_seed_partial.json'
for _si,_s in enumerate(SEEDS):
    print(f'\n========== SEED {_s}  ({_si+1}/{len(SEEDS)}) ==========')
    _rec={'seed':_s,'longtail':{}}
    for _vname,_vflags in GNN_VARIANTS:
        _cmd=_common(_vflags)+f' --seed {_s}'
        print(f' {_vname}:',_cmd); get_ipython().system(_cmd)
        assert os.path.exists('CPGRec_plus/data_exist/final_embeddings.pth'), f'{_vname} seed {_s}: sin embeddings (driver murio)'
        if app_id_mapping is None:
            with open('CPGRec_plus/data_exist/app_id_mapping.pkl','rb') as f: app_id_mapping=_pk.load(f)
            with open('CPGRec_plus/data_exist/user_id_mapping.pkl','rb') as f: user_id_mapping=_pk.load(f)
            idx2app_cpg={v:int(k) for k,v in app_id_mapping.items()}
        _emb=_t.load('CPGRec_plus/data_exist/final_embeddings.pth',map_location='cpu')
        _r,_=recs_emb(_emb['user'].numpy(),_emb['game'].numpy(),user_id_mapping,idx2app_cpg,eval_set,TOPN)
        _rec[_vname]=_clean(paper_metrics(_r,test_items_per_user,cat_maps=CAT_MAPS))
        _rec['longtail'][_vname]=_longtail(_r)
        if _si==0: seed0_recs[_vname]=_r
        print(f'   -> {_vname}: R@5={_rec[_vname]["Recall@5"]:.4f} NDCG@5={_rec[_vname]["NDCG@5"]:.4f}')
        del _emb
    per_seed.append(_rec)
    _json.dump(per_seed, open(_PARTIAL,'w'), indent=2)
    print(f'[seed {_s}] base R@5={_rec["CPGRec base"]["Recall@5"]:.4f} | '
          f'+PER R@5={_rec["CPGRec +PER"]["Recall@5"]:.4f} | '
          f'+PER+PRG R@5={_rec["CPGRec +PER+PRG"]["Recall@5"]:.4f}   (parcial -> {_PARTIAL})')
print(f'\nMulti-seed completo: {len(per_seed)}/{len(SEEDS)} seeds.')


========== SEED 42  (1/3) ==========
 CPGRec base: cd CPGRec_plus && python run_cpgrec.py --path ./steam_data --data_exist ./data_exist --use_per 0 --use_llm 0 --freeze_user_llm 0 --or_cap 50 --or_weight size --neg_mode random --neg_aux_weight 1 --epochs 1000 --embed_size 32 --lr 0.03 --m 6.5 --gamma 80.0 --param_decay 0.1 --seed 42
INFO:root:device = cuda:0
INFO:root:build train data:
100% 62944/62944 [00:00<00:00, 96276.55it/s]
INFO:root:build valid data:
100% 57429/57429 [00:00<00:00, 216956.33it/s]
INFO:root:build test data:
100% 57390/57390 [00:00<00:00, 212898.85it/s]
INFO:root:read app info:
/content/CPGRec_plus/utils/dataloader_steam.py:463: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  dates_timestamps = dates.view(np.int64)
INFO:root:reading genre,developer,publisher info...
100% 18380/18380 [00:00<00:00, 1959118.36it/s]
100% 18380/18380 [00:00<00:00, 1274131.19it/s]
100% 7449/7449 [0

/tmp/ipykernel_7019/938798465.py:52: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  _emb=_t.load('CPGRec_plus/data_exist/final_embeddings.pth',map_location='cpu')


   -> CPGRec base: R@5=0.1440 NDCG@5=0.2028
 CPGRec +PER: cd CPGRec_plus && python run_cpgrec.py --path ./steam_data --data_exist ./data_exist --use_per 1 --use_llm 0 --freeze_user_llm 0 --or_cap 50 --or_weight size --neg_mode random --neg_aux_weight 1 --epochs 1000 --embed_size 32 --lr 0.03 --m 6.5 --gamma 80.0 --param_decay 0.1 --seed 42
INFO:root:device = cuda:0
INFO:root:build train data:
INFO:root:build valid data:
INFO:root:build test data:
INFO:root:read app info:
INFO:root:reading genre,developer,publisher info...
INFO:root:reading user item play time...
/content/CPGRec_plus/utils/dataloader_steam.py:335: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default 

## E. Modelos feature-based (FM / DeepFM / DeepNN) — full-ranking comparable (Opción B)

Re-implementación de los 3 modelos *deep* de Cheuque **bajo el protocolo full-ranking de este comparativo**, para que sus filas sean **directamente comparables** con ALS/CPGRec (mismas métricas, mismo `eval_set`, mismo split 5-core).

**Diferencias clave vs. `cheuque_ucsd_inicial.ipynb`:**
- **Sin fuga de `playtime`**: el full-ranking puntúa pares (usuario, ítem) no observados, donde no existe `playtime` → se elimina como feature (era la fuente del NDCG≈0,99 en Cheuque).
- **Eval = full-ranking del catálogo** (no re-ranking por-usuario): top-N sobre todo `CATALOG`, enmascarando lo visto en train → `paper_metrics(..., CAT_MAPS)` + `_longtail`, idéntico a ALS/CPGRec.
- **Entrenamiento pointwise**: positivos implícitos de `train` + negative sampling (BCE), porque deepctr-torch es un clasificador binario (CPGRec es BPR-style).
- **Multi-seed** (mismos `SEEDS` que el GNN) → media±std en las tablas.

Las filas se integran automáticamente en las tablas A (accuracy), B (diversidad), C (long-tail) y D (ejemplos).

In [ ]:
# ====== E. FM / DeepFM / DeepNN -- full-ranking comparable (Opcion B) : features + helpers ======
# Mismos datos 5-core, mismo eval_set y MISMO eval full-ranking que ALS/CPGRec.
# SIN fuga de playtime. deepctr-torch es pointwise binario -> positivos implicitos + neg-sampling.
get_ipython().system('pip install -q deepctr-torch --no-deps')
import torch, time as _time
from deepctr_torch.inputs import SparseFeat, DenseFeat, VarLenSparseFeat, get_feature_names
from deepctr_torch.models import DeepFM, WDL

DEVICE_DEEP = 'cuda' if torch.cuda.is_available() else 'cpu'
EMB_DEEP    = 32
DEEP_NEG    = 4                                  # negativos por positivo (BCE)
DEEP_BATCH  = 16384                              # batch grande -> menos overhead del DataLoader de deepctr
# Presupuesto por tier (deepctr es lento sobre millones de filas):
#  T2 (final): cap de positivos de ENTRENAMIENTO (no del eval) para caber en la sesion; eval COMPLETO -> filas comparables.
#  T1 (sanity): todo chico, valida la fontaneria en minutos.
DEEP_EPOCHS = 10        if TIER == 'T2' else 1
DEEP_MAXPOS = 2_000_000 if TIER == 'T2' else 200_000     # tope de positivos de train (None = todos)
DEEP_UBATCH = 1024                              # usuarios por lote en el scoring full-ranking

N_USERS_D = int(inter['user_id'].max()) + 1      # user_id factorizado 0..N-1
item2idx  = dict(_a2i)                            # app_id -> 0..n_catalog-1 (MISMO indice que ALS)
idx2app_d = {i: a for a, i in item2idx.items()}
N_ITEMS_D = len(item2idx)

# --- dense (validas para CUALQUIER par u,i; NINGUNA usa playtime) ---
# 'count' es nivel-usuario -> constante intra-usuario, no altera el ranking; se mantiene por paridad con Cheuque.
_uc = train.groupby('user_id')['app_id'].size()
user_count = np.zeros(N_USERS_D, dtype='float32'); user_count[_uc.index.to_numpy()] = _uc.to_numpy().astype('float32')
_rvd = reviews.copy(); _rvd['app_id'] = pd.to_numeric(_rvd['app_id'], errors='coerce')
_rvd = _rvd.dropna(subset=['app_id']); _rvd['app_id'] = _rvd['app_id'].astype('int64')
_rcd = _rvd.assign(_r=_rvd['recommend'].fillna(False).astype(bool)).groupby('app_id')['_r'].sum()
item_reccount = np.array([float(_rcd.get(a, 0.0)) for a in CATALOG], dtype='float32')
_meta = dict(zip(cat['app_id'].astype(int), cat['metascore'].astype(float)))   # metascore = positive_ratio (rating de PER)
item_meta = np.array([float(_meta.get(a, 0.0)) for a in CATALOG], dtype='float32')

def _z(x, logx=True):
    x = np.log1p(np.clip(x, 0, None)) if logx else x.astype('float32')
    return ((x - x.mean()) / (x.std() + 1e-6)).astype('float32')
user_count_z    = _z(user_count)
item_reccount_z = _z(item_reccount)
item_meta_z     = _z(item_meta, logx=False)

# --- generos -> secuencia padded por item (item-level) ---
_allg = sorted({g for a in CATALOG for g in genre_map.get(a, [])})
genre2id = {g: k + 1 for k, g in enumerate(_allg)}      # 0 = padding
VOCAB_G, MAXLEN_G = len(_allg) + 1, 3
def _padg(a):
    ids = [genre2id[g] for g in genre_map.get(a, []) if g in genre2id][:MAXLEN_G]
    return ids + [0] * (MAXLEN_G - len(ids))
item_gseq = np.stack([_padg(a) for a in CATALOG]).astype('int64')        # (N_ITEMS, MAXLEN_G)
item_glen = np.array([max(1, min(len(genre_map.get(a, [])), MAXLEN_G)) for a in CATALOG], dtype='int64')

DENSE_COLS_D = ['count', 'reccount', 'metascore']
def _featcols_d():
    sparse = [SparseFeat('user_idx', N_USERS_D, embedding_dim=EMB_DEEP),
              SparseFeat('item_idx', N_ITEMS_D, embedding_dim=EMB_DEEP)]
    varlen = [VarLenSparseFeat(SparseFeat('genres_seq', VOCAB_G, embedding_dim=EMB_DEEP),
                               maxlen=MAXLEN_G, combiner='mean', length_name='genres_len')]
    dense  = [DenseFeat(c, 1) for c in DENSE_COLS_D]
    cols = sparse + dense + varlen
    return cols, cols

def _build_train_d(seed, max_pos=None):
    """positivos = interacciones de train (submuestreadas a max_pos si aplica) ; negativos = DEEP_NEG ~ uniforme."""
    if max_pos is None: max_pos = DEEP_MAXPOS
    rng = np.random.default_rng(seed)
    tr = train[train['app_id'].isin(item2idx)]
    pu = tr['user_id'].to_numpy().astype('int64')
    pi = tr['app_id'].map(item2idx).to_numpy().astype('int64')
    if max_pos is not None and len(pu) > max_pos:
        sel = rng.choice(len(pu), size=max_pos, replace=False)
        pu, pi = pu[sel], pi[sel]
    P  = len(pu)
    nu = np.repeat(pu, DEEP_NEG)
    ni = rng.integers(0, N_ITEMS_D, size=P * DEEP_NEG).astype('int64')
    u  = np.concatenate([pu, nu]); i = np.concatenate([pi, ni])
    y  = np.concatenate([np.ones(P, 'float32'), np.zeros(P * DEEP_NEG, 'float32')])
    pm = rng.permutation(len(u)); u, i, y = u[pm], i[pm], y[pm]
    X  = {'user_idx': u, 'item_idx': i, 'count': user_count_z[u],
          'reccount': item_reccount_z[i], 'metascore': item_meta_z[i],
          'genres_seq': item_gseq[i], 'genres_len': item_glen[i]}
    return X, y

def _build_model_d(kind, lin, dnn):
    if kind == 'FM':     return DeepFM(lin, dnn, dnn_hidden_units=(),       task='binary', device=DEVICE_DEEP)
    if kind == 'DeepFM': return DeepFM(lin, dnn, dnn_hidden_units=(8, 8), dnn_dropout=0.2, task='binary', device=DEVICE_DEEP)
    if kind == 'DeepNN': return WDL([],  dnn, dnn_hidden_units=(32, 32), dnn_dropout=0.2, task='binary', device=DEVICE_DEEP)
    raise ValueError(kind)

def _deep_full_rank(model, users, topn, ubatch=None, log_every=20):
    """Full-ranking: por cada usuario puntua TODO el catalogo, enmascara train, top-N. Imprime progreso."""
    if ubatch is None: ubatch = DEEP_UBATCH
    out = {}; all_i = np.arange(N_ITEMS_D, dtype='int64')
    nb = (len(users) + ubatch - 1) // ubatch
    for bi_, s in enumerate(range(0, len(users), ubatch)):
        ub = np.asarray(users[s:s + ubatch], dtype='int64'); B = len(ub)
        u_rep = np.repeat(ub, N_ITEMS_D); i_rep = np.tile(all_i, B)
        X = {'user_idx': u_rep, 'item_idx': i_rep, 'count': user_count_z[u_rep],
             'reccount': item_reccount_z[i_rep], 'metascore': item_meta_z[i_rep],
             'genres_seq': item_gseq[i_rep], 'genres_len': item_glen[i_rep]}
        sc = model.predict(X, batch_size=131072).reshape(B, N_ITEMS_D)
        for bi in range(B):
            u = int(ub[bi]); seen = train_items_per_user.get(u, set()); rec = []
            for j in np.argsort(-sc[bi]):
                a = idx2app_d[int(j)]
                if a not in seen:
                    rec.append(a)
                    if len(rec) >= topn: break
            out[u] = rec
        if log_every and (bi_ % log_every == 0):
            print(f'       full-rank lote {bi_+1}/{nb}', flush=True)
    return out

def _longtail_deep(recs_u, users):
    """Como _longtail pero itera SOLO los usuarios puntuados (para soportar eval cap en T1)."""
    out = {}
    for b in buckets:
        us = [u for u in users if _act.get(u) == b and test_items_per_user.get(u) and u in recs_u]
        if not us: continue
        out[b] = {'n': len(us),
                  'NDCG@10':  float(np.mean([_u_nr(recs_u[u], test_items_per_user[u], 10)[0] for u in us])),
                  'Recall@10':float(np.mean([_u_nr(recs_u[u], test_items_per_user[u], 10)[1] for u in us]))}
    return out

print(f'deep listo: N_USERS={N_USERS_D} N_ITEMS={N_ITEMS_D} generos={len(_allg)} | TIER={TIER} '
      f'epochs={DEEP_EPOCHS} neg={DEEP_NEG} maxpos={DEEP_MAXPOS} batch={DEEP_BATCH} device={DEVICE_DEEP}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.1 MB/s eta 0:00:00
deep listo: N_USERS=62944 N_ITEMS=9198 generos=21 | TIER=T2 epochs=10 neg=4 maxpos=2000000 batch=16384 device=cuda


In [23]:
# ====== E. Loop multi-seed FM/DeepFM/DeepNN (entrena + full-ranking + metricas; CON progreso) ======
import json as _json, time as _time
DEEP_VARIANTS = ['FM', 'DeepFM', 'DeepNN']
DEEP_SEEDS = SEEDS if TIER == 'T2' else SEEDS[:1]                       # T1 sanity = 1 seed
DEEP_EVAL  = list(eval_set) if TIER == 'T2' else list(eval_set)[:2000]  # T2 = eval COMPLETO (comparable); T1 = 2000
_lin_d, _dnn_d = _featcols_d()
deep_per_seed = []; deep_seed0_recs = {}
os.makedirs('comparativo_ucsd_corregido', exist_ok=True)
_PARTIAL_D = 'comparativo_ucsd_corregido/deep_per_seed_partial.json'
print(f'DEEP run: TIER={TIER} seeds={DEEP_SEEDS} epochs={DEEP_EPOCHS} neg={DEEP_NEG} '
      f'maxpos={DEEP_MAXPOS} eval_users={len(DEEP_EVAL):,} batch={DEEP_BATCH}', flush=True)
for _si, _s in enumerate(DEEP_SEEDS):
    print(f'\n========== DEEP SEED {_s}  ({_si+1}/{len(DEEP_SEEDS)}) ==========', flush=True)
    torch.manual_seed(_s); np.random.seed(_s)
    _Xtr, _ytr = _build_train_d(_s)
    print(f'  train: {len(_ytr):,} muestras ({int(_ytr.sum()):,} pos)', flush=True)
    _rec = {'seed': _s, 'longtail': {}}
    for _k in DEEP_VARIANTS:
        torch.manual_seed(_s)
        _m = _build_model_d(_k, _lin_d, _dnn_d)
        _m.compile('adam', 'binary_crossentropy', metrics=[])
        print(f'  [{_k}] entrenando {DEEP_EPOCHS} ep (batch {DEEP_BATCH})...', flush=True)
        for _ep in range(DEEP_EPOCHS):
            _t0 = _time.time()
            _h = _m.fit(_Xtr, _ytr, batch_size=DEEP_BATCH, epochs=1, verbose=0, shuffle=True)
            _ls = _h.history.get('loss', [float('nan')])[-1]
            print(f'     {_k} ep {_ep+1}/{DEEP_EPOCHS}  loss={_ls:.4f}  ({_time.time()-_t0:.0f}s)', flush=True)
        print(f'  [{_k}] full-ranking sobre {len(DEEP_EVAL):,} usuarios...', flush=True)
        _r = _deep_full_rank(_m, DEEP_EVAL, TOPN)
        _rec[_k] = _clean(paper_metrics(_r, test_items_per_user, cat_maps=CAT_MAPS))
        _rec['longtail'][_k] = _longtail_deep(_r, DEEP_EVAL)
        if _si == 0: deep_seed0_recs[_k] = _r
        print(f'   -> {_k:7s}: R@5={_rec[_k]["Recall@5"]:.4f} NDCG@5={_rec[_k]["NDCG@5"]:.4f} '
              f'NDCG@10={_rec[_k]["NDCG@10"]:.4f} Cov_tot@10={_rec[_k]["Cov_total@10"]:.2f}', flush=True)
        del _m
        if DEVICE_DEEP == 'cuda': torch.cuda.empty_cache()
    deep_per_seed.append(_rec)
    _json.dump(deep_per_seed, open(_PARTIAL_D, 'w'), indent=2)
    print(f'[deep seed {_s}] FM R@5={_rec["FM"]["Recall@5"]:.4f} | '
          f'DeepFM R@5={_rec["DeepFM"]["Recall@5"]:.4f} | DeepNN R@5={_rec["DeepNN"]["Recall@5"]:.4f}  '
          f'(parcial -> {_PARTIAL_D})', flush=True)
print(f'\nDeep multi-seed completo: {len(deep_per_seed)}/{len(DEEP_SEEDS)} seeds.')


DEEP run: TIER=T2 seeds=[42, 1, 2] epochs=10 neg=4 maxpos=2000000 eval_users=57,390 batch=16384

========== DEEP SEED 42  (1/3) ==========
  train: 10,000,000 muestras (2,000,000 pos)
  [FM] entrenando 10 ep (batch 16384)...
cuda
Train on 10000000 samples, validate on 0 samples, 611 steps per epoch
     FM ep 1/10  loss=0.3596  (135s)
cuda
Train on 10000000 samples, validate on 0 samples, 611 steps per epoch
     FM ep 2/10  loss=0.2756  (137s)
cuda
Train on 10000000 samples, validate on 0 samples, 611 steps per epoch
     FM ep 3/10  loss=0.2640  (136s)
cuda
Train on 10000000 samples, validate on 0 samples, 611 steps per epoch
     FM ep 4/10  loss=0.2539  (135s)
cuda
Train on 10000000 samples, validate on 0 samples, 611 steps per epoch
     FM ep 5/10  loss=0.2461  (134s)
cuda
Train on 10000000 samples, validate on 0 samples, 611 steps per epoch
     FM ep 6/10  loss=0.2399  (135s)
cuda
Train on 10000000 samples, validate on 0 samples, 611 steps per epoch
     FM ep 7/10  loss=0.2347

## A. Accuracy (media +- std + deltas pareados)

In [24]:
# ====== A. Accuracy (media+-std sobre seeds) + deltas pareados (PER aislado, modelo completo) ======
GNN_MODELS=['CPGRec base','CPGRec +PER','CPGRec +PER+PRG']
def _agg(model):
    return {c:(float(np.mean([ps[model][c] for ps in per_seed])),
               float(np.std([ps[model][c] for ps in per_seed]))) for c in ACC}
agg={m:_agg(m) for m in GNN_MODELS}
rowsA={}
rowsA['Most Popular']={c:f'{metrics_fixed["Most Popular"][c]:.4f}' for c in ACC}
rowsA['ALS']={c:f'{metrics_fixed["ALS"][c]:.4f}' for c in ACC}
for m in GNN_MODELS:
    rowsA[m]={c:f'{agg[m][c][0]:.4f}±{agg[m][c][1]:.4f}' for c in ACC}
if 'deep_per_seed' in globals() and deep_per_seed:
    for m in ['FM','DeepFM','DeepNN']:
        _ad={c:(float(np.mean([ps[m][c] for ps in deep_per_seed])),float(np.std([ps[m][c] for ps in deep_per_seed]))) for c in ACC}
        rowsA[m]={c:f'{_ad[c][0]:.4f}±{_ad[c][1]:.4f}' for c in ACC}
for k,v in REPORTED.items():
    rowsA[k]={c:(f'{v.get(c):.4f}' if v.get(c) is not None else '') for c in ACC}
dfA=pd.DataFrame(rowsA).T[ACC]
print(f'=== A Accuracy (n_eval={len(eval_set):,} | {len(per_seed)} seeds; media±std) ===\n')
print(dfA.to_string())
# Deltas pareados (mismo split/eval/seed): PER aislado, modelo completo, PRG sobre PER.
_DELTAS=[('+PER vs base',     'CPGRec +PER',     'CPGRec base'),
         ('+PER+PRG vs base', 'CPGRec +PER+PRG', 'CPGRec base'),
         ('+PER+PRG vs +PER', 'CPGRec +PER+PRG', 'CPGRec +PER')]
delA={}
for _lab,_hi,_lo in _DELTAS:
    print(f'\n--- Delta pareado ({_lab}) por seed ---')
    delA[_lab]={}
    for c in ACC:
        d=[ps[_hi][c]-ps[_lo][c] for ps in per_seed]
        delA[_lab][c]={'mean':float(np.mean(d)),'std':float(np.std(d)),'pos':int(sum(1 for x in d if x>0)),'n':len(d)}
        print(f'  {c:13s}: d={np.mean(d):+.4f} +- {np.std(d):.4f}   ({delA[_lab][c]["pos"]}/{delA[_lab][c]["n"]} seeds con d>0)')

=== A Accuracy (n_eval=57,390 | 3 seeds; media±std) ===

                      Recall@5         NDCG@5          Hit@5    Precision@5      Recall@10        NDCG@10         Hit@10   Precision@10
Most Popular            0.1170         0.1652         0.4635         0.1244         0.1599         0.1633         0.5717         0.0900
ALS                     0.0868         0.0910         0.2774         0.0655         0.1407         0.1101         0.4333         0.0586
CPGRec base      0.1440±0.0011  0.2030±0.0016  0.5401±0.0023  0.1565±0.0012  0.2043±0.0013  0.2046±0.0015  0.6561±0.0026  0.1166±0.0009
CPGRec +PER      0.1377±0.0031  0.1943±0.0040  0.5237±0.0084  0.1500±0.0033  0.1970±0.0045  0.1966±0.0043  0.6438±0.0090  0.1127±0.0027
CPGRec +PER+PRG  0.1411±0.0009  0.1980±0.0009  0.5306±0.0020  0.1525±0.0008  0.2004±0.0010  0.2002±0.0009  0.6490±0.0023  0.1144±0.0006
FM               0.0905±0.0013  0.1333±0.0015  0.3978±0.0029  0.1046±0.0009  0.1371±0.0016  0.1375±0.0015  0.5252±0.0031  0.082

## B. Diversidad

In [25]:
# ====== B. Diversidad (Cov/Ent categoria; media+-std multi-seed) ======
DIV=[f'Cov_total@{k}' for k in KS]+[f'Cov_gene@{k}' for k in KS]+[f'Ent_gene@{k}' for k in KS]
def _aggd(model):
    return {c:(float(np.mean([ps[model][c] for ps in per_seed])),
               float(np.std([ps[model][c] for ps in per_seed]))) for c in DIV}
rowsB={}
rowsB['Most Popular']={c:f'{metrics_fixed["Most Popular"][c]:.4f}' for c in DIV}
rowsB['ALS']={c:f'{metrics_fixed["ALS"][c]:.4f}' for c in DIV}
for m in ('CPGRec base','CPGRec +PER','CPGRec +PER+PRG'):
    a=_aggd(m); rowsB[m]={c:f'{a[c][0]:.4f}±{a[c][1]:.4f}' for c in DIV}
if 'deep_per_seed' in globals() and deep_per_seed:
    for m in ['FM','DeepFM','DeepNN']:
        _bd={c:(float(np.mean([ps[m][c] for ps in deep_per_seed])),float(np.std([ps[m][c] for ps in deep_per_seed]))) for c in DIV}
        rowsB[m]={c:f'{_bd[c][0]:.4f}±{_bd[c][1]:.4f}' for c in DIV}
dfB=pd.DataFrame(rowsB).T[DIV]
print('=== B Diversidad (Cov=nro categorias distintas en top-K; Ent=entropia; media±std) ===\n')
print(dfB.to_string())

=== B Diversidad (Cov=nro categorias distintas en top-K; Ent=entropia; media±std) ===

                    Cov_total@5    Cov_total@10     Cov_gene@5    Cov_gene@10     Ent_gene@5    Ent_gene@10
Most Popular            11.1864         17.5005         5.4129         7.0011         2.1893         2.4478
ALS                     13.0900         21.9564         5.5699         7.3716         2.1822         2.5070
CPGRec base      11.8881±0.1709  19.5041±0.1543  5.3668±0.1256  6.8419±0.0973  2.1121±0.0398  2.3800±0.0275
CPGRec +PER      11.4485±0.2443  19.1343±0.4798  5.2527±0.1357  6.7873±0.1772  2.0754±0.0467  2.3604±0.0494
CPGRec +PER+PRG  11.4545±0.1516  19.0483±0.1185  5.1820±0.0904  6.7518±0.0297  2.0612±0.0231  2.3575±0.0108
FM               10.7157±0.0731  18.3308±0.0802  4.2120±0.0434  5.7024±0.0580  1.6400±0.0268  1.9772±0.0276
DeepFM           10.1972±0.2020  17.6603±0.2455  4.0267±0.0574  5.5962±0.0339  1.5795±0.0355  1.9518±0.0197
DeepNN            8.9604±0.3826  16.7790±0.6383  

## C. Long-tail (por actividad del usuario)

In [26]:
# ====== C. Long-tail: NDCG@10 por actividad (media+-std multi-seed) ======
lt_fixed={m:_longtail(recs_fixed[m]) for m in recs_fixed}
order=['Most Popular','ALS','CPGRec base','CPGRec +PER','CPGRec +PER+PRG']
_DEEP_LT=['FM','DeepFM','DeepNN'] if ('deep_per_seed' in globals() and deep_per_seed) else []
order=order+_DEEP_LT
rowsC=[]; nrow={}
for b in buckets:
    row={'actividad':b}
    for m in ('Most Popular','ALS'):
        row[m]=(f'{lt_fixed[m][b]["NDCG@10"]:.4f}' if b in lt_fixed[m] else '')
    for m in ('CPGRec base','CPGRec +PER','CPGRec +PER+PRG'):
        vals=[ps['longtail'][m][b]['NDCG@10'] for ps in per_seed if b in ps['longtail'][m]]
        row[m]=(f'{np.mean(vals):.4f}±{np.std(vals):.4f}' if vals else '')
    for m in _DEEP_LT:
        vals=[ps['longtail'][m][b]['NDCG@10'] for ps in deep_per_seed if b in ps['longtail'][m]]
        row[m]=(f'{np.mean(vals):.4f}±{np.std(vals):.4f}' if vals else '')
    rowsC.append(row)
    nrow[b]=lt_fixed['Most Popular'].get(b,{}).get('n','?')
dfC=pd.DataFrame(rowsC)[['actividad']+order]
print('=== C Long-tail: NDCG@10 por actividad (media±std) ===')
print(dfC.to_string(index=False))
print('\nn usuarios por bucket:')
for b in buckets: print(f'  {b:6s}: {nrow[b]}')

=== C Long-tail: NDCG@10 por actividad (media±std) ===
actividad Most Popular    ALS   CPGRec base   CPGRec +PER CPGRec +PER+PRG            FM        DeepFM        DeepNN
      2-5       0.2206 0.1953 0.2208±0.0016 0.2170±0.0005   0.2230±0.0028 0.1539±0.0107 0.1634±0.0079 0.0547±0.0018
     6-20       0.1740 0.1841 0.2133±0.0013 0.2037±0.0039   0.2088±0.0012 0.1297±0.0030 0.1331±0.0032 0.0422±0.0004
    21-50       0.1547 0.1219 0.1967±0.0016 0.1872±0.0052   0.1918±0.0008 0.1274±0.0016 0.1279±0.0015 0.0513±0.0014
      51+       0.1605 0.0597 0.2051±0.0018 0.1986±0.0041   0.2006±0.0009 0.1476±0.0009 0.1525±0.0006 0.1137±0.0007

n usuarios por bucket:
  2-5   : 1750
  6-20  : 12122
  21-50 : 18396
  51+   : 25122


## D. Ejemplos de recomendaciones (poster)

In [27]:
# ====== D. Ejemplos de recomendaciones (seed representativo = primer seed) ======
recs={'Most Popular':recs_fixed['Most Popular'],'ALS':recs_fixed['ALS'],
      'CPGRec base':seed0_recs['CPGRec base'],'CPGRec +PER':seed0_recs['CPGRec +PER'],
      'CPGRec +PER+PRG':seed0_recs['CPGRec +PER+PRG']}
if 'deep_seed0_recs' in globals() and deep_seed0_recs:
    for m in ['FM','DeepFM','DeepNN']:
        if m in deep_seed0_recs and len(deep_seed0_recs[m])>=len(eval_set): recs[m]=deep_seed0_recs[m]
def _nm(items): return [name_map.get(a,str(a)) for a in items]
_chosen=[]
for b in buckets:
    for u in [x for x in eval_set if _act[x]==b and test_items_per_user.get(x)]:
        if any(any(it in test_items_per_user[u] for it in recs[m][u][:5]) for m in recs):
            _chosen.append((b,u)); break
    if len(_chosen)>=N_EXAMPLES: break
ej=[f'(ejemplos del seed {SEEDS[0]})']
for b,u in _chosen[:N_EXAMPLES]:
    rel=test_items_per_user[u]; hist=sorted(train_items_per_user.get(u,set()))
    ej.append(f'\n### Usuario {user_code2orig.get(u, u)}  (actividad {b}, {len(hist)} juegos en historial)')
    ej.append('- Perfil (muestra): ' + ', '.join(_nm(hist[:6])))
    ej.append('- Test (a acertar): ' + ', '.join(_nm(sorted(rel))))
    for m in recs:
        nd,rc=_u_nr(recs[m][u],rel,5)
        marks=[name_map.get(a,str(a))+(' ✓' if a in rel else '') for a in recs[m][u][:5]]
        ej.append(f'  - **{m}** (R@5={rc:.2f}, NDCG@5={nd:.2f}): ' + ', '.join(marks))
ej_txt='\n'.join(ej)
print('=== D Ejemplos ===\n'+ej_txt)

=== D Ejemplos ===
(ejemplos del seed 42)

### Usuario 011111135489484797  (actividad 2-5, 1 juegos en historial)
- Perfil (muestra): app_205790
- Test (a acertar): Garry's Mod, app_223530
  - **Most Popular** (R@5=1.00, NDCG@5=0.65): Counter-Strike: Global Offensive, Garry's Mod ✓, Unturned, app_223530 ✓, Left 4 Dead 2
  - **ALS** (R@5=0.50, NDCG@5=0.26): Counter-Strike: Global Offensive, Left 4 Dead 2, Unturned, Garry's Mod ✓, Warframe
  - **CPGRec base** (R@5=0.50, NDCG@5=0.31): Counter-Strike: Global Offensive, Unturned, Garry's Mod ✓, Left 4 Dead 2, Robocraft
  - **CPGRec +PER** (R@5=1.00, NDCG@5=0.65): Unturned, Garry's Mod ✓, Counter-Strike: Global Offensive, app_223530 ✓, Left 4 Dead 2
  - **CPGRec +PER+PRG** (R@5=0.50, NDCG@5=0.31): Counter-Strike: Global Offensive, Unturned, Garry's Mod ✓, Left 4 Dead 2, Robocraft
  - **FM** (R@5=0.50, NDCG@5=0.61): app_223530 ✓, PAYDAY 2, War Thunder, app_72850, app_33910
  - **DeepFM** (R@5=1.00, NDCG@5=0.85): app_223530 ✓, PAYDAY 2, War Th

## Guardar + imprimir todo (respaldo sin descarga)

In [28]:
# ====== Guardar (chicos) + IMPRIMIR TODO (respaldo si no se pueden descargar) ======
import json as _json
_out='comparativo_ucsd_corregido'; os.makedirs(_out, exist_ok=True)
dfA.to_csv(f'{_out}/accuracy.csv'); dfB.to_csv(f'{_out}/diversidad.csv'); dfC.to_csv(f'{_out}/longtail.csv', index=False)
open(f'{_out}/ejemplos.md','w',encoding='utf-8').write(ej_txt)
_full={'seeds':SEEDS,'n_eval':len(eval_set),'per_seed':per_seed,
       'fixed':{m:{k:(float(v) if isinstance(v,(int,float,np.floating)) else v) for k,v in metrics_fixed[m].items()} for m in metrics_fixed},
       'delta':delA}
if 'deep_per_seed' in globals() and deep_per_seed: _full['deep_per_seed']=deep_per_seed
_json.dump(_full, open(f'{_out}/metricas_full.json','w'), indent=2)
print('Guardado en', _out, '->', sorted(os.listdir(_out)))
print('\n'+'#'*72)
print('# RESPALDO EN TEXTO  (si no se descargan los archivos, todo queda impreso aqui)')
print('#'*72)
print('\n===== accuracy.csv =====\n'+dfA.to_csv())
print('\n===== diversidad.csv =====\n'+dfB.to_csv())
print('\n===== longtail.csv =====\n'+dfC.to_csv(index=False))
print('\n===== ejemplos.md =====\n'+ej_txt)
print('\n===== metricas_full.json =====\n'+_json.dumps(_full, indent=2))
try:
    import shutil as _s2; _s2.make_archive(_out,'zip',_out)
    from google.colab import files; files.download(_out+'.zip')
    print('\n(zip de descarga generado)')
except Exception as e:
    print('\n(descarga automatica no disponible:', repr(e), '-> todo esta impreso arriba)')


Guardado en comparativo_ucsd_corregido -> ['accuracy.csv', 'deep_per_seed_partial.json', 'diversidad.csv', 'ejemplos.md', 'longtail.csv', 'metricas_full.json', 'per_seed_partial.json']

########################################################################
# RESPALDO EN TEXTO  (si no se descargan los archivos, todo queda impreso aqui)
########################################################################

===== accuracy.csv =====
,Recall@5,NDCG@5,Hit@5,Precision@5,Recall@10,NDCG@10,Hit@10,Precision@10
Most Popular,0.1170,0.1652,0.4635,0.1244,0.1599,0.1633,0.5717,0.0900
ALS,0.0868,0.0910,0.2774,0.0655,0.1407,0.1101,0.4333,0.0586
CPGRec base,0.1440±0.0011,0.2030±0.0016,0.5401±0.0023,0.1565±0.0012,0.2043±0.0013,0.2046±0.0015,0.6561±0.0026,0.1166±0.0009
CPGRec +PER,0.1377±0.0031,0.1943±0.0040,0.5237±0.0084,0.1500±0.0033,0.1970±0.0045,0.1966±0.0043,0.6438±0.0090,0.1127±0.0027
CPGRec +PER+PRG,0.1411±0.0009,0.1980±0.0009,0.5306±0.0020,0.1525±0.0008,0.2004±0.0010,0.2002±0.0009,0.6490±0.002

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


(zip de descarga generado)
